In [11]:
"""
Mock API Simulator
==================

A self-contained local HTTP API simulator using only Python's standard library
plus the external ``requests`` package for the client-side tests.

Endpoints
---------
GET  /api/users
GET  /api/users/<id>
POST /api/users

By default the server listens on port 4221. Override with ``--port`` or the
``MOCK_API_PORT`` environment variable.

Usage
-----
    python mock_api.py                 # :4221 + smoke tests
    python mock_api.py --no-tests      # server only
    python mock_api.py --port 8080
    MOCK_API_PORT=9000 python mock_api.py -v
"""

from __future__ import annotations

import argparse
import json
import logging
import os
import socket
import sys
import threading
import time
from http.server import BaseHTTPRequestHandler, HTTPServer
from socketserver import ThreadingMixIn
from typing import Any, Optional
from urllib.parse import urlsplit

import requests


# ============================================================
# Configuration
# ============================================================

HOST = "localhost"
DEFAULT_PORT = 4221
REQUEST_TIMEOUT = 5
MAX_BODY_BYTES = 1_000_000  # 1 MB cap on incoming request bodies

log = logging.getLogger("mockapi")


# ============================================================
# Mock Database
# ============================================================

_INITIAL_USERS = [
    {"id": 1, "name": "Alice", "role": "Admin"},
    {"id": 2, "name": "Bob", "role": "User"},
]

Mock_DB: dict[str, list[dict[str, Any]]] = {
    "users": [dict(u) for u in _INITIAL_USERS],
}

# RLock keeps us safe even if a helper calls another lock-acquiring helper.
_db_lock = threading.RLock()


def reset_database() -> None:
    with _db_lock:
        Mock_DB["users"].clear()
        Mock_DB["users"].extend(dict(u) for u in _INITIAL_USERS)


def next_user_id() -> int:
    """Return the next available user ID. Caller must not hold a stale view."""
    with _db_lock:
        if not Mock_DB["users"]:
            return 1
        return max(u["id"] for u in Mock_DB["users"]) + 1


def get_user_by_id(user_id: int) -> Optional[dict[str, Any]]:
    """Return a *copy* of the user, or ``None`` if not found."""
    with _db_lock:
        for user in Mock_DB["users"]:
            if user["id"] == user_id:
                return dict(user)
    return None


def list_users() -> list[dict[str, Any]]:
    with _db_lock:
        return [dict(u) for u in Mock_DB["users"]]


def add_user(name: str, role: str) -> dict[str, Any]:
    with _db_lock:
        new_user = {"id": next_user_id(), "name": name, "role": role}
        Mock_DB["users"].append(new_user)
        return dict(new_user)


# ============================================================
# API Request Handler
# ============================================================

class MockAPIHandler(BaseHTTPRequestHandler):
    """Routes the mock API. Each method must write exactly one response."""

    protocol_version = "HTTP/1.1"
    server_version = "MockAPI/1.1"

    # ---------------- response helpers ----------------

    def _send_json(self, payload: Any, status_code: int = 200) -> None:
        body = json.dumps(payload).encode("utf-8")
        self.send_response(status_code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Connection", "close")
        self.end_headers()
        try:
            self.wfile.write(body)
        except (BrokenPipeError, ConnectionResetError):
            # Client hung up before we could finish; nothing to do.
            pass
        self.close_connection = True

    def _send_error_json(
        self, status: int, error: str, message: str
    ) -> None:
        self._send_json({"error": error, "message": message}, status)

    def _method_not_allowed(self, *allowed: str) -> None:
        body = json.dumps(
            {
                "error": "Method Not Allowed",
                "message": f"Allowed methods: {', '.join(allowed)}",
            }
        ).encode("utf-8")
        self.send_response(405)
        self.send_header("Allow", ", ".join(allowed))
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Connection", "close")
        self.end_headers()
        try:
            self.wfile.write(body)
        except (BrokenPipeError, ConnectionResetError):
            pass
        self.close_connection = True

    # ---------------- request parsing ----------------

    def _get_path(self) -> str:
        return urlsplit(self.path).path

    def _read_json_object(self) -> dict[str, Any]:
        """Read and validate the JSON body. Raises ValueError on any problem."""
        raw_length = self.headers.get("Content-Length")
        if raw_length is None:
            raise ValueError("Content-Length header is required")
        try:
            length = int(raw_length)
        except ValueError as exc:
            raise ValueError("Content-Length must be an integer") from exc
        if length < 0:
            raise ValueError("Content-Length must be non-negative")
        if length > MAX_BODY_BYTES:
            raise ValueError(
                f"Request body exceeds limit of {MAX_BODY_BYTES} bytes"
            )

        raw = self.rfile.read(length)
        try:
            parsed = json.loads(raw.decode("utf-8"))
        except (UnicodeDecodeError, json.JSONDecodeError) as exc:
            raise ValueError(f"Invalid JSON: {exc}") from exc
        if not isinstance(parsed, dict):
            raise ValueError("Request body must be a JSON object")
        return parsed

    # ---------------- routes ----------------

    def do_GET(self) -> None:
        path = self._get_path()
        try:
            if path == "/api/users":
                self._send_json(list_users(), 200)
                return

            if path.startswith("/api/users/"):
                suffix = path[len("/api/users/"):]
                try:
                    user_id = int(suffix)
                except ValueError:
                    self._send_error_json(
                        400, "Bad Request", "User ID must be an integer"
                    )
                    return

                user = get_user_by_id(user_id)
                if user is None:
                    self._send_error_json(
                        404,
                        "Not Found",
                        f"User with ID {user_id} not found",
                    )
                    return
                self._send_json(user, 200)
                return

            self._send_error_json(404, "Not Found", f"No route for {path}")
        except Exception:  # pragma: no cover - defensive
            log.exception("Unhandled error in do_GET")
            self._send_error_json(
                500, "Internal Server Error", "Unexpected server error"
            )

    def do_POST(self) -> None:
        path = self._get_path()
        if path != "/api/users":
            self._send_error_json(404, "Not Found", f"No route for {path}")
            return

        try:
            payload = self._read_json_object()
            name = payload.get("name")
            role = payload.get("role")

            if not isinstance(name, str) or not name.strip():
                self._send_error_json(
                    400,
                    "Invalid input",
                    "'name' is required and must be a non-empty string",
                )
                return
            if not isinstance(role, str) or not role.strip():
                self._send_error_json(
                    400,
                    "Invalid input",
                    "'role' is required and must be a non-empty string",
                )
                return

            created = add_user(name.strip(), role.strip())
            self._send_json(created, 201)
        except ValueError as exc:
            self._send_error_json(400, "Bad Request", str(exc))
        except Exception:  # pragma: no cover - defensive
            log.exception("Unhandled error in do_POST")
            self._send_error_json(
                500, "Internal Server Error", "Unexpected server error"
            )

    def do_PUT(self) -> None:  # noqa: N802
        self._method_not_allowed("GET", "POST")

    do_DELETE = do_PATCH = do_PUT  # type: ignore[assignment]

    def log_message(self, fmt: str, *args: Any) -> None:
        log.debug("%s - %s", self.address_string(), fmt % args)


# ============================================================
# HTTP Server
# ============================================================

class ThreadingHTTPServer(ThreadingMixIn, HTTPServer):
    """HTTPServer that handles each request on its own daemon thread."""

    allow_reuse_address = True
    daemon_threads = True

    def handle_error(
        self, request: socket.socket, client_address: Any
    ) -> None:
        exc = sys.exc_info()[1]
        if isinstance(exc, (BrokenPipeError, ConnectionResetError)):
            return
        super().handle_error(request, client_address)


def run_server(
    port: int,
    ready_event: Optional[threading.Event] = None,
    stop_event: Optional[threading.Event] = None,
) -> None:
    """Bind, listen, and serve until ``stop_event`` is set."""
    server = ThreadingHTTPServer(("", port), MockAPIHandler)

    if ready_event is not None:
        ready_event.set()

    log.info("Listening on http://%s:%d", HOST, port)

    if stop_event is not None:
        def _watch_stop() -> None:
            stop_event.wait()
            server.shutdown()

        threading.Thread(
            target=_watch_stop, name="server-shutdown", daemon=True
        ).start()

    try:
        server.serve_forever(poll_interval=0.2)
    finally:
        server.server_close()
        log.info("Server stopped")


# ============================================================
# Client Helpers
# ============================================================

def api_url(port: int, path: str) -> str:
    return f"http://{HOST}:{port}{path}"


def get_users(port: int) -> requests.Response:
    return requests.get(
        api_url(port, "/api/users"), timeout=REQUEST_TIMEOUT
    )


def get_user(port: int, user_id: int) -> requests.Response:
    return requests.get(
        api_url(port, f"/api/users/{user_id}"), timeout=REQUEST_TIMEOUT
    )


def create_user(port: int, name: str, role: str) -> requests.Response:
    return requests.post(
        api_url(port, "/api/users"),
        json={"name": name, "role": role},
        timeout=REQUEST_TIMEOUT,
    )


def print_response(response: requests.Response) -> None:
    print("Status Code:", response.status_code)
    try:
        print("Response:", response.json())
    except ValueError:
        print("Response:", response.text)


def list_users_as_table(api_port: int) -> None:
    try:
        response = get_users(api_port)
    except requests.RequestException as exc:
        print(f"An error occurred while fetching users: {exc}")
        return

    if response.status_code != 200:
        print(
            f"Failed to fetch users. Status code: {response.status_code}"
        )
        return

    users = response.json()
    headers = ["ID", "Name", "Role"]
    col_widths = [6, 20, 15]

    row_format = (
        "| " + " | ".join(f"{{:<{w}}}" for w in col_widths) + " |"
    )
    border = "+" + "+".join("-" * (w + 2) for w in col_widths) + "+"

    print(border)
    print(row_format.format(*headers))
    print(border)
    for user in users:
        print(
            row_format.format(
                user.get("id", ""),
                user.get("name", ""),
                user.get("role", ""),
            )
        )
    print(border)


# ============================================================
# Demo / Test Functions
# ============================================================

def test_get_existing_user(port: int) -> None:
    print("\n--- Testing GET /api/users/1 ---")
    print_response(get_user(port, 1))


def test_get_nonexistent_user(port: int) -> None:
    print("\n--- Testing GET /api/users/99 ---")
    print_response(get_user(port, 99))


def test_get_invalid_user_id(port: int) -> None:
    print("\n--- Testing GET /api/users/invalid_id ---")
    print_response(
        requests.get(
            api_url(port, "/api/users/invalid_id"),
            timeout=REQUEST_TIMEOUT,
        )
    )


def test_create_user(port: int) -> None:
    print("\n--- Testing POST /api/users ---")
    print_response(create_user(port, "Charlie", "Developer"))


def test_invalid_user(port: int) -> None:
    print("\n--- Testing POST /api/users with invalid data ---")
    print_response(create_user(port, "", "User"))


def test_missing_name(port: int) -> None:
    print("\n--- Testing POST /api/users with missing name ---")
    print_response(
        requests.post(
            api_url(port, "/api/users"),
            json={"role": "User"},
            timeout=REQUEST_TIMEOUT,
        )
    )


def test_missing_role(port: int) -> None:
    print("\n--- Testing POST /api/users with missing role ---")
    print_response(
        requests.post(
            api_url(port, "/api/users"),
            json={"name": "Eve"},
            timeout=REQUEST_TIMEOUT,
        )
    )


def test_invalid_json(port: int) -> None:
    print("\n--- Testing POST /api/users with invalid JSON ---")
    print_response(
        requests.post(
            api_url(port, "/api/users"),
            data="{not valid json",
            headers={"Content-Type": "application/json"},
            timeout=REQUEST_TIMEOUT,
        )
    )


def test_unsupported_method(port: int) -> None:
    print("\n--- Testing DELETE /api/users/1 (method not allowed) ---")
    print_response(
        requests.delete(
            api_url(port, "/api/users/1"), timeout=REQUEST_TIMEOUT
        )
    )


# ============================================================
# Automated Assertions
# ============================================================

def assert_status(
    response: requests.Response, expected: int
) -> None:
    assert response.status_code == expected, (
        f"Expected {expected}, got {response.status_code}: "
        f"{response.text!r}"
    )


def run_assertion_tests(port: int) -> None:
    print("\n--- Running automated API assertions ---")

    response = get_users(port)
    assert_status(response, 200)
    assert isinstance(response.json(), list)

    response = get_user(port, 1)
    assert_status(response, 200)
    assert response.json()["name"] == "Alice"

    response = get_user(port, 9999)
    assert_status(response, 404)

    response = requests.get(
        api_url(port, "/api/users/not-an-int"),
        timeout=REQUEST_TIMEOUT,
    )
    assert_status(response, 400)

    response = create_user(port, "Test User", "Tester")
    assert_status(response, 201)
    assert response.json()["name"] == "Test User"

    response = create_user(port, "", "Tester")
    assert_status(response, 400)

    # New: unsupported method returns 405 with an Allow header.
    response = requests.delete(
        api_url(port, "/api/users/1"), timeout=REQUEST_TIMEOUT
    )
    assert_status(response, 405)
    assert "GET" in response.headers.get("Allow", "")

    print("All automated API assertions passed.")


# ============================================================
# CLI / Entry Point
# ============================================================

def _parse_args(argv: Optional[list[str]] = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Local mock HTTP API simulator.",
    )
    parser.add_argument(
        "--port",
        type=int,
        default=None,
        help=(
            "Port to listen on. Defaults to MOCK_API_PORT, "
            f"then {DEFAULT_PORT}."
        ),
    )
    parser.add_argument(
        "--no-tests",
        action="store_true",
        help="Start the server without running the demo suite.",
    )
    parser.add_argument(
        "-v",
        "--verbose",
        action="store_true",
        help="Enable DEBUG-level logging.",
    )
    return parser.parse_args(argv)


def _resolve_port(cli_port: Optional[int]) -> int:
    if cli_port is not None:
        return cli_port
    env = os.environ.get("MOCK_API_PORT")
    if env:
        try:
            return int(env)
        except ValueError as exc:
            raise SystemExit(
                f"MOCK_API_PORT must be an integer, got {env!r}"
            ) from exc
    return DEFAULT_PORT


def _run_demo(port: int) -> None:
    print("=" * 60)
    print("MOCK API TEST")
    print("=" * 60)

    test_get_existing_user(port)
    test_get_nonexistent_user(port)
    test_get_invalid_user_id(port)

    print("\n--- Testing GET /api/users ---")
    list_users_as_table(port)

    test_create_user(port)

    print("\n--- Users After Creating Charlie ---")
    list_users_as_table(port)

    test_invalid_user(port)
    test_missing_name(port)
    test_missing_role(port)
    test_invalid_json(port)
    test_unsupported_method(port)

    print("\n--- Final User List ---")
    list_users_as_table(port)

    run_assertion_tests(port)


def main(argv: Optional[list[str]] = None) -> int:
    args = _parse_args(argv)

    logging.basicConfig(
        level=logging.DEBUG if args.verbose else logging.INFO,
        format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    )

    reset_database()
    port = _resolve_port(args.port)

    ready = threading.Event()
    stop = threading.Event()

    server_thread = threading.Thread(
        target=run_server,
        args=(port, ready, stop),
        name="mock-api-server",
        daemon=True,
    )
    server_thread.start()

    if not ready.wait(timeout=5):
        log.error("Server failed to start within 5 seconds")
        return 1

    if not args.no_tests:
        try:
            _run_demo(port)
        except Exception:
            log.exception("Demo suite crashed")
            stop.set()
            server_thread.join(timeout=5)
            return 2

    log.info(
        "API simulator is running at http://%s:%d (Ctrl+C to stop)",
        HOST,
        port,
    )

    try:
        server_thread.join()
    except KeyboardInterrupt:
        log.info("Shutting down...")
    finally:
        stop.set()
        server_thread.join(timeout=5)

    return 0


if __name__ == "__main__":
    # If we are running in an interactive notebook context (IPython/Colab),
    # we safely bypass sys.argv parsing which is meant for shell CLI use.
    if 'ipykernel' in sys.modules or 'google.colab' in sys.modules:
        # Start on port 4221 by default, or find an open port.
        import socket
        def get_free_port():
            s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            s.bind(('', 0))
            p = s.getsockname()[1]
            s.close()
            return p

        port = get_free_port()
        reset_database()
        ready = threading.Event()
        server_thread = threading.Thread(
            target=run_server,
            args=(port, ready),
            name="mock-api-server",
            daemon=True,
        )
        server_thread.start()
        if ready.wait(timeout=5):
            _run_demo(port)
    else:
        sys.exit(main())


MOCK API TEST

--- Testing GET /api/users/1 ---
Status Code: 200
Response: {'id': 1, 'name': 'Alice', 'role': 'Admin'}

--- Testing GET /api/users/99 ---
Status Code: 404
Response: {'error': 'Not Found', 'message': 'User with ID 99 not found'}

--- Testing GET /api/users/invalid_id ---
Status Code: 400
Response: {'error': 'Bad Request', 'message': 'User ID must be an integer'}

--- Testing GET /api/users ---
+--------+----------------------+-----------------+
| ID     | Name                 | Role            |
+--------+----------------------+-----------------+
| 1      | Alice                | Admin           |
| 2      | Bob                  | User            |
+--------+----------------------+-----------------+

--- Testing POST /api/users ---
Status Code: 201
Response: {'id': 3, 'name': 'Charlie', 'role': 'Developer'}

--- Users After Creating Charlie ---
+--------+----------------------+-----------------+
| ID     | Name                 | Role            |
+--------+--------------

In [13]:
"""
Mock API Simulator (v1.3.0)
===========================

A self-contained local HTTP API simulator built on Python's standard
library, plus the external ``requests`` package for the client-side
smoke tests.

Endpoints
---------
GET    /                       Service index (name, version, link list)
GET    /health                 Liveness probe (status, version, uptime)
GET    /api/stats              Aggregate request + data statistics
GET    /openapi.json           OpenAPI 3.0 document for the whole API

GET    /api/users              List users (supports ?page, ?per_page,
                               ?role, ?q filtering / pagination)
GET    /api/users/<id>         Fetch a single user
POST   /api/users              Create a user
PUT    /api/users/<id>         Replace a user (full update)
PATCH  /api/users/<id>         Partially update a user
DELETE /api/users/<id>         Delete a user

Every response carries an ``X-Request-ID`` header (echoed when the
client sends one, generated otherwise) for tracing.

``HEAD`` mirrors ``GET`` (headers only), ``OPTIONS`` returns ``204`` with
an ``Allow`` header, and all responses include permissive CORS headers
so browser-based test harnesses can call the mock directly.

Configuration
-------------
By default the server listens on port 4221. Override with ``--port`` or
the ``MOCK_API_PORT`` environment variable.

Usage
-----
    python mock_api.py                 # :4221 + smoke tests
    python mock_api.py --no-tests      # server only
    python mock_api.py --port 8080
    MOCK_API_PORT=9000 python mock_api.py -v
"""

from __future__ import annotations

import argparse
import copy
import json
import logging
import os
import re
import socket
import sys
import threading
import time
import uuid
from datetime import datetime, timezone
from http.server import BaseHTTPRequestHandler, HTTPServer
from socketserver import ThreadingMixIn
from typing import Any, Callable, NamedTuple, Optional
from urllib.parse import parse_qs, urlsplit

import requests


# ============================================================
# Configuration
# ============================================================

HOST = "localhost"
DEFAULT_PORT = 4221
REQUEST_TIMEOUT = 5
MAX_BODY_BYTES = 1_000_000  # 1 MB cap on incoming request bodies
API_VERSION = "1.3.0"

# Validation limits.
MAX_NAME_LEN = 100
MAX_ROLE_LEN = 50
ALLOWED_ROLES = {"Admin", "User", "Developer", "Maintainer", "Guest"}

# Pagination defaults and bounds.
DEFAULT_PAGE_SIZE = 10
MAX_PAGE_SIZE = 100

# CORS allow-lists (permissive; override in a real deployment).
CORS_ALLOW_ORIGIN = "*"
CORS_ALLOW_METHODS = "GET, HEAD, POST, PUT, PATCH, DELETE, OPTIONS"
CORS_ALLOW_HEADERS = "Content-Type, X-Request-ID"
CORS_EXPOSE_HEADERS = "X-Request-ID, Allow"

log = logging.getLogger("mockapi")

# Set by run_server() so /openapi.json can advertise the real bound port.
_BOUND_PORT: Optional[int] = None


# ============================================================
# Mock Database
# ============================================================

_INITIAL_USERS = [
    {"id": 1, "name": "Alice", "role": "Admin"},
    {"id": 2, "name": "Bob", "role": "User"},
]

# Kept as a public name for backward compatibility; treat as private.
Mock_DB: dict[str, list[dict[str, Any]]] = {
    "users": [dict(u) for u in _INITIAL_USERS],
}

# RLock keeps us safe even if a helper calls another lock-acquiring helper.
_db_lock = threading.RLock()


def reset_database() -> None:
    with _db_lock:
        Mock_DB["users"].clear()
        Mock_DB["users"].extend(dict(u) for u in _INITIAL_USERS)


def next_user_id() -> int:
    with _db_lock:
        if not Mock_DB["users"]:
            return 1
        return max(u["id"] for u in Mock_DB["users"]) + 1


def get_user_by_id(user_id: int) -> Optional[dict[str, Any]]:
    """Return a *copy* of the user, or ``None`` if not found."""
    with _db_lock:
        for user in Mock_DB["users"]:
            if user["id"] == user_id:
                return dict(user)
    return None


def list_users() -> list[dict[str, Any]]:
    with _db_lock:
        return [dict(u) for u in Mock_DB["users"]]


def add_user(name: str, role: str) -> dict[str, Any]:
    with _db_lock:
        new_user = {"id": next_user_id(), "name": name, "role": role}
        Mock_DB["users"].append(new_user)
        return dict(new_user)


def update_user(user_id: int, name: str, role: str) -> Optional[dict[str, Any]]:
    with _db_lock:
        for user in Mock_DB["users"]:
            if user["id"] == user_id:
                user["name"] = name
                user["role"] = role
                return dict(user)
    return None


def patch_user(
    user_id: int, *, name: Optional[str] = None, role: Optional[str] = None
) -> Optional[dict[str, Any]]:
    with _db_lock:
        for user in Mock_DB["users"]:
            if user["id"] == user_id:
                if name is not None:
                    user["name"] = name
                if role is not None:
                    user["role" ] = role
                return dict(user)
    return None


def delete_user(user_id: int) -> bool:
    with _db_lock:
        for i, user in enumerate(Mock_DB["users"]):
            if user["id"] == user_id:
                Mock_DB["users"].pop(i)
                return True
    return False


# ============================================================
# Request Statistics
# ============================================================

_stats_lock = threading.Lock()
_stats: dict[str, Any] = {
    "started_at": datetime.now(timezone.utc).isoformat(),
    "start_perf": time.monotonic(),
    "requests_total": 0,
    "by_method": {},
    "by_path": {},
}


def _bump_stats(method: str, path: str) -> None:
    with _stats_lock:
        _stats["requests_total"] += 1
        _stats["by_method"][method] = _stats["by_method"].get(method, 0) + 1
        _stats["by_path"][path] = _stats["by_path"].get(path, 0) + 1


def _snapshot_stats() -> dict[str, Any]:
    with _stats_lock:
        return {
            "requests_total": _stats["requests_total"],
            "by_method": dict(_stats["by_method"]),
            "by_path": dict(_stats["by_path"]),
            "started_at": _stats["started_at"],
            "uptime_seconds": round(time.monotonic() - _stats["start_perf"], 3),
        }


def _reset_stats() -> None:
    with _stats_lock:
        _stats["started_at"] = datetime.now(timezone.utc).isoformat()
        _stats["start_perf"] = time.monotonic()
        _stats["requests_total"] = 0
        _stats["by_method"] = {}
        _stats["by_path"] = {}


# ============================================================
# Validation
# ============================================================

class UnsupportedMediaType(ValueError):
    """Raised when the request's Content-Type is not application/json."""

    status = 415


class ValidatedUser(NamedTuple):
    """Result of validate_user_payload().

    On success ``error`` is ``None``. ``name``/``role`` are ``str`` when
    ``require_all=True``; for PATCH (``require_all=False``) they are
    ``None`` when the field was not supplied.
    """

    name: Optional[str]
    role: Optional[str]
    error: Optional[str]


def validate_user_payload(
    payload: dict[str, Any], require_all: bool = True
) -> ValidatedUser:
    """Validate a user payload.

    When ``require_all`` is False (PATCH semantics) only present fields
    are validated and missing ones come back as ``None`` without error.
    """
    name = payload.get("name")
    role = payload.get("role")

    if require_all:
        if not isinstance(name, str) or not name.strip() or len(name) > MAX_NAME_LEN:
            return ValidatedUser(
                None,
                None,
                f"'name' is required and must be a non-empty "
                f"string of at most {MAX_NAME_LEN} characters",
            )
        if not isinstance(role, str) or not role.strip() or len(role) > MAX_ROLE_LEN:
            return ValidatedUser(
                None,
                None,
                f"'role' is required and must be a non-empty "
                f"string of at most {MAX_ROLE_LEN} characters",
            )
        if role.strip() not in ALLOWED_ROLES:
            return ValidatedUser(
                None,
                None,
                f"'role' must be one of {sorted(ALLOWED_ROLES)}",
            )
        return ValidatedUser(name.strip(), role.strip(), None)

    # PATCH: validate only provided fields.
    clean_name: Optional[str] = None
    clean_role: Optional[str] = None
    if name is not None:
        if not isinstance(name, str) or not name.strip() or len(name) > MAX_NAME_LEN:
            return ValidatedUser(
                None,
                None,
                f"'name' must be a non-empty string of at most "
                f"{MAX_NAME_LEN} characters",
            )
        clean_name = name.strip()
    if role is not None:
        if not isinstance(role, str) or not role.strip() or len(role) > MAX_ROLE_LEN:
            return ValidatedUser(
                None,
                None,
                f"'role' must be a non-empty string of at most "
                f"{MAX_ROLE_LEN} characters",
            )
        if role.strip() not in ALLOWED_ROLES:
            return ValidatedUser(
                None, None, f"'role' must be one of {sorted(ALLOWED_ROLES)}"
            )
        clean_role = role.strip()
    return ValidatedUser(clean_name, clean_role, None)


# ============================================================
# Router
# ============================================================

class Route:
    """A single (method, regex) -> handler binding."""

    __slots__ = ("method", "pattern", "handler")

    def __init__(
        self, method: str, pattern: str, handler: Callable[..., None]
    ) -> None:
        self.method = method
        self.pattern = re.compile(pattern)
        self.handler = handler


ROUTES: list[Route] = []


def route(
    method: str, pattern: str
) -> Callable[[Callable[..., None]], Callable[..., None]]:
    """Register a handler for ``method`` matching ``pattern`` (anchored)."""

    def decorator(fn: Callable[..., None]) -> Callable[..., None]:
        ROUTES.append(Route(method, pattern, fn))
        return fn

    return decorator


def _allowed_methods_for(path: str) -> list[str]:
    methods = {r.method for r in ROUTES if r.pattern.match(path)}
    # HEAD is always implied by GET.
    if "GET" in methods:
        methods.add("HEAD")
    return sorted(methods)


def _parse_int(qs: dict[str, list[str]], key: str, default: int) -> int:
    values = qs.get(key)
    if not values:
        return default
    try:
        return int(values[0])
    except ValueError as exc:
        raise ValueError(f"'{key}' must be an integer") from exc


# ============================================================
# API Request Handler
# ============================================================

class MockAPIHandler(BaseHTTPRequestHandler):
    """Routes the mock API. Each method must write exactly one response."""

    # We always send ``Connection: close``, so advertise HTTP/1.0 honestly.
    protocol_version = "HTTP/1.0"
    server_version = f"MockAPI/{API_VERSION}"

    # ---------------- response helpers ----------------

    def _send_cors_headers(self) -> None:
        self.send_header("Access-Control-Allow-Origin", CORS_ALLOW_ORIGIN)
        self.send_header("Access-Control-Allow-Methods", CORS_ALLOW_METHODS)
        self.send_header("Access-Control-Allow-Headers", CORS_ALLOW_HEADERS)
        self.send_header("Access-Control-Expose-Headers", CORS_EXPOSE_HEADERS)

    def _send_common_headers(self) -> None:
        """Headers shared by every response we emit."""
        self.send_header("Connection", "close")
        rid = getattr(self, "_request_id", None)
        if rid:
            self.send_header("X-Request-ID", rid)
        self._send_cors_headers()

    def _send_json(
        self,
        payload: Any,
        status_code: int = 200,
        extra_headers: Optional[dict[str, str]] = None,
    ) -> None:
        body = json.dumps(payload).encode("utf-8")
        self.send_response(status_code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self._send_common_headers()
        for k, v in (extra_headers or {}).items():
            self.send_header(k, v)
        self.end_headers()
        # HEAD: send headers only, but with the length the body would have.
        if self.command != "HEAD":
            try:
                self.wfile.write(body)
            except (BrokenPipeError, ConnectionResetError):
                pass
        self.close_connection = True

    def _send_no_content(
        self,
        status_code: int = 204,
        extra_headers: Optional[dict[str, str]] = None,
    ) -> None:
        self.send_response(status_code)
        self.send_header("Content-Length", "0")
        self._send_common_headers()
        for k, v in (extra_headers or {}).items():
            self.send_header(k, v)
        self.end_headers()
        self.close_connection = True

    def _send_error_json(
        self, status: int, error: str, message: str
    ) -> None:
        self._send_json({"error": error, "message": message}, status)

    def _method_not_allowed(self, *allowed: str) -> None:
        self._send_json(
            {
                "error": "Method Not Allowed",
                "message": f"Allowed methods: {', '.join(allowed)}",
            },
            405,
            extra_headers={"Allow": ", ".join(allowed)},
        )

    # ---------------- request parsing ----------------

    def _get_path(self) -> str:
        return urlsplit(self.path).path

    def _get_query(self) -> dict[str, list[str]]:
        return parse_qs(urlsplit(self.path).query, keep_blank_values=True)

    def _ensure_request_id(self) -> str:
        rid = self.headers.get("X-Request-ID") or uuid.uuid4().hex
        self._request_id = rid
        return rid

    def _read_json_object(self) -> dict[str, Any]:
        """Read and validate the JSON body.

        Raises ``UnsupportedMediaType`` for a wrong Content-Type and
        ``ValueError`` for any body-level problem.
        """
        ctype = self.headers.get("Content-Type", "")
        if ctype and "application/json" not in ctype.lower():
            raise UnsupportedMediaType("Content-Type must be application/json")

        raw_length = self.headers.get("Content-Length")
        if raw_length is None:
            raise ValueError("Content-Length header is required")
        try:
            length = int(raw_length)
        except ValueError as exc:
            raise ValueError("Content-Length must be an integer") from exc
        if length < 0:
            raise ValueError("Content-Length must be non-negative")
        if length > MAX_BODY_BYTES:
            raise ValueError(
                f"Request body exceeds limit of {MAX_BODY_BYTES} bytes"
            )

        raw = self.rfile.read(length)
        try:
            parsed = json.loads(raw)
        except (UnicodeDecodeError, json.JSONDecodeError) as exc:
            raise ValueError(f"Invalid JSON: {exc}") from exc
        if not isinstance(parsed, dict):
            raise ValueError("Request body must be a JSON object")
        return parsed

    # ---------------- dispatch ----------------

    def _dispatch(self, method: str) -> None:
        path = self._get_path()
        query = self._get_query()
        request_id = self._ensure_request_id()
        _bump_stats(method, path)
        log.debug("[%s] %s %s", request_id, method, path)

        try:
            if method == "OPTIONS":
                allowed = _allowed_methods_for(path)
                if not allowed:
                    self._send_error_json(404, "Not Found", f"No route for {path}")
                    return
                self._send_no_content(204, extra_headers={"Allow": ", ".join(allowed)})
                return

            lookup = "GET" if method == "HEAD" else method
            for r in ROUTES:
                if r.method != lookup:
                    continue
                m = r.pattern.match(path)
                if m:
                    r.handler(self, m, query)
                    return

            allowed = _allowed_methods_for(path)
            if allowed:
                self._method_not_allowed(*allowed)
            else:
                self._send_error_json(404, "Not Found", f"No route for {path}")
        except UnsupportedMediaType as exc:
            self._send_error_json(415, "Unsupported Media Type", str(exc))
        except ValueError as exc:
            self._send_error_json(400, "Bad Request", str(exc))
        except Exception:  # pragma: no cover - defensive
            log.exception("[%s] Unhandled error", request_id)
            self._send_error_json(
                500, "Internal Server Error", "Unexpected server error"
            )

    # Standard HTTP verbs all funnel through _dispatch.
    def do_GET(self) -> None:  # noqa: N802
        self._dispatch("GET")

    def do_HEAD(self) -> None:  # noqa: N802
        self._dispatch("HEAD")

    def do_POST(self) -> None:  # noqa: N802
        self._dispatch("POST")

    def do_PUT(self) -> None:  # noqa: N802
        self._dispatch("PUT")

    def do_PATCH(self) -> None:  # noqa: N802
        self._dispatch("PATCH")

    def do_DELETE(self) -> None:  # noqa: N802
        self._dispatch("DELETE")

    def do_OPTIONS(self) -> None:  # noqa: N802
        self._dispatch("OPTIONS")

    def log_message(self, fmt: str, *args: Any) -> None:
        log.debug("%s - %s", self.address_string(), fmt % args)


# ============================================================
# Route Handlers
# ============================================================
#
# Registration order matters: integer-id routes (``\d+``) must be
# registered before the catch-all bad-id routes (``[^/]+``) so that a
# numeric suffix is matched by the right handler.

@route("GET", r"^/api/users/?$")
def handle_list_users(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    users = list_users()

    # parse_qs(keep_blank_values=True) turns ``?role=`` into [""];
    # the truthiness check treats that as "no filter", which is intended.
    role = (qs.get("role") or [None])[0]
    search = (qs.get("q") or [None])[0]
    if role:
        users = [u for u in users if u["role"].lower() == role.lower()]
    if search:
        needle = search.lower()
        users = [u for u in users if needle in u["name"].lower()]

    has_paging = any(k in qs for k in ("page", "per_page"))
    if has_paging:
        page = _parse_int(qs, "page", 1)
        per_page = _parse_int(qs, "per_page", DEFAULT_PAGE_SIZE)
        if page < 1:
            raise ValueError("'page' must be >= 1")
        if per_page < 1 or per_page > MAX_PAGE_SIZE:
            raise ValueError(
                f"'per_page' must be between 1 and {MAX_PAGE_SIZE}"
            )
        total = len(users)
        total_pages = (total + per_page - 1) // per_page if total else 0
        start = (page - 1) * per_page
        page_items = users[start:start + per_page]
        h._send_json(
            {
                "data": page_items,
                "page": page,
                "per_page": per_page,
                "total": total,
                "total_pages": total_pages,
            },
            200,
        )
    else:
        h._send_json(users, 200)


@route("GET", r"^/api/users/(\d+)/?$")
def handle_get_user(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    user_id = int(m.group(1))
    user = get_user_by_id(user_id)
    if user is None:
        h._send_error_json(404, "Not Found", f"User with ID {user_id} not found")
        return
    h._send_json(user, 200)


@route("GET", r"^/api/users/([^/]+)/?$")
def handle_get_user_bad_id(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    h._send_error_json(400, "Bad Request", "User ID must be an integer")


@route("POST", r"^/api/users/?$")
def handle_create_user(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    payload = h._read_json_object()
    name, role, err = validate_user_payload(payload, require_all=True)
    if err:
        h._send_error_json(400, "Invalid input", err)
        return
    assert name is not None and role is not None  # narrowed by validation
    created = add_user(name, role)
    h._send_json(created, 201)


@route("PUT", r"^/api/users/(\d+)/?$")
def handle_replace_user(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    user_id = int(m.group(1))
    if get_user_by_id(user_id) is None:
        h._send_error_json(404, "Not Found", f"User with ID {user_id} not found")
        return
    payload = h._read_json_object()
    name, role, err = validate_user_payload(payload, require_all=True)
    if err:
        h._send_error_json(400, "Invalid input", err)
        return
    assert name is not None and role is not None
    updated = update_user(user_id, name, role)
    h._send_json(updated, 200)


@route("PUT", r"^/api/users/([^/]+)/?$")
def handle_replace_user_bad_id(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    h._send_error_json(400, "Bad Request", "User ID must be an integer")


@route("PATCH", r"^/api/users/(\d+)/?$")
def handle_patch_user(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    user_id = int(m.group(1))
    if get_user_by_id(user_id) is None:
        h._send_error_json(404, "Not Found", f"User with ID {user_id} not found")
        return
    payload = h._read_json_object()
    name, role, err = validate_user_payload(payload, require_all=False)
    if err:
        h._send_error_json(400, "Invalid input", err)
        return
    if name is None and role is None:
        h._send_error_json(
            400,
            "Invalid input",
            "No valid fields to update; supply 'name' and/or 'role'",
        )
        return
    updated = patch_user(user_id, name=name, role=role)
    h._send_json(updated, 200)


@route("PATCH", r"^/api/users/([^/]+)/?$")
def handle_patch_user_bad_id(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    h._send_error_json(400, "Bad Request", "User ID must be an integer")


@route("DELETE", r"^/api/users/(\d+)/?$")
def handle_delete_user(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    user_id = int(m.group(1))
    if delete_user(user_id):
        h._send_json({"deleted": True, "id": user_id}, 200)
    else:
        h._send_error_json(404, "Not Found", f"User with ID {user_id} not found")


@route("DELETE", r"^/api/users/([^/]+)/?$")
def handle_delete_user_bad_id(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    h._send_error_json(400, "Bad Request", "User ID must be an integer")


@route("GET", r"^/health/?$")
def handle_health(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    # Route through the locked snapshot so a concurrent _reset_stats()
    # cannot yield a stale / negative uptime.
    snapshot = _snapshot_stats()
    h._send_json(
        {
            "status": "ok",
            "version": API_VERSION,
            "uptime_seconds": snapshot["uptime_seconds"],
        },
        200,
    )


@route("GET", r"^/api/stats/?$")
def handle_stats(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    snapshot = _snapshot_stats()
    users = list_users()  # single copy
    roles: dict[str, int] = {}
    for u in users:
        roles[u["role"]] = roles.get(u["role"], 0) + 1
    snapshot["user_count"] = len(users)
    snapshot["roles"] = roles
    h._send_json(snapshot, 200)


@route("GET", r"^/openapi\.json/?$")
def handle_openapi(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    # Serve a copy so mutating ``servers`` never touches the template.
    spec = copy.deepcopy(OPENAPI_SPEC)
    if _BOUND_PORT is not None:
        spec["servers"] = [{"url": f"http://{HOST}:{_BOUND_PORT}"}]
    h._send_json(spec, 200)


@route("GET", r"^/?$")
def handle_root(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    h._send_json(
        {
            "name": "Mock API",
            "version": API_VERSION,
            "endpoints": [
                "GET /api/users",
                "GET /api/users/<id>",
                "POST /api/users",
                "PUT /api/users/<id>",
                "PATCH /api/users/<id>",
                "DELETE /api/users/<id>",
                "GET /health",
                "GET /api/stats",
                "GET /openapi.json",
            ],
        },
        200,
    )


# ============================================================
# OpenAPI 3.0 Document
# ============================================================

OPENAPI_SPEC: dict[str, Any] = {
    "openapi": "3.0.3",
    "info": {
        "title": "Mock API",
        "version": API_VERSION,
        "description": "A self-contained local mock HTTP API simulator.",
    },
    # Placeholder; /openapi.json replaces this with the actual bound port.
    "servers": [{"url": f"http://{HOST}:{DEFAULT_PORT}"}],
    "paths": {
        "/": {
            "get": {
                "summary": "Service index",
                "responses": {"200": {"description": "API metadata"}},
            }
        },
        "/api/users": {
            "get": {
                "summary": "List users",
                "description": (
                    "Returns all users as a bare list. Add `page` or "
                    "`per_page` to get a paginated envelope; filter with "
                    "`role` and search with `q`."
                ),
                "parameters": [
                    {
                        "name": "page",
                        "in": "query",
                        "schema": {"type": "integer", "minimum": 1},
                    },
                    {
                        "name": "per_page",
                        "in": "query",
                        "schema": {
                            "type": "integer",
                            "minimum": 1,
                            "maximum": MAX_PAGE_SIZE,
                        },
                    },
                    {"name": "role", "in": "query", "schema": {"type": "string"}},
                    {"name": "q", "in": "query", "schema": {"type": "string"}},
                ],
                "responses": {
                    "200": {
                        "description": "User list or paginated envelope",
                        "content": {"application/json": {"schema": {}}},
                    }
                },
            },
            "post": {
                "summary": "Create a user",
                "requestBody": {
                    "required": True,
                    "content": {
                        "application/json": {
                            "schema": {"$ref": "#/components/schemas/UserCreate"}
                        }
                    },
                },
                "responses": {
                    "201": {"description": "Created user"},
                    "400": {"description": "Invalid input"},
                    "415": {"description": "Unsupported Media Type"},
                },
            },
        },
        "/api/users/{id}": {
            "get": {
                "summary": "Fetch a user",
                "responses": {
                    "200": {"description": "The user"},
                    "404": {"description": "Not found"},
                },
            },
            "put": {
                "summary": "Replace a user",
                "responses": {
                    "200": {"description": "Updated user"},
                    "400": {"description": "Invalid input"},
                    "404": {"description": "Not found"},
                },
            },
            "patch": {
                "summary": "Partially update a user",
                "responses": {
                    "200": {"description": "Updated user"},
                    "400": {"description": "Invalid input"},
                    "404": {"description": "Not found"},
                },
            },
            "delete": {
                "summary": "Delete a user",
                "responses": {
                    "200": {"description": "Deleted"},
                    "404": {"description": "Not found"},
                },
            },
        },
        "/health": {
            "get": {
                "summary": "Liveness probe",
                "responses": {"200": {"description": "Service health"}},
            }
        },
        "/api/stats": {
            "get": {
                "summary": "Request and data statistics",
                "responses": {"200": {"description": "Statistics snapshot"}},
            }
        },
        "/openapi.json": {
            "get": {
                "summary": "OpenAPI document",
                "responses": {"200": {"description": "OpenAPI 3.0 JSON"}},
            }
        },
    },
    "components": {
        "schemas": {
            "UserCreate": {
                "type": "object",
                "required": ["name", "role"],
                "properties": {
                    "name": {"type": "string", "maxLength": MAX_NAME_LEN},
                    "role": {"type": "string", "enum": sorted(ALLOWED_ROLES)},
                },
            },
            "User": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer"},
                    "name": {"type": "string"},
                    "role": {"type": "string"},
                },
            },
        }
    },
}


# ============================================================
# HTTP Server
# ============================================================

class ThreadingHTTPServer(ThreadingMixIn, HTTPServer):
    """HTTPServer that handles each request on its own daemon thread."""

    allow_reuse_address = True
    daemon_threads = True

    def handle_error(
        self, request: socket.socket, client_address: Any
    ) -> None:
        exc = sys.exc_info()[1]
        if isinstance(exc, (BrokenPipeError, ConnectionResetError)):
            return
        super().handle_error(request, client_address)


def run_server(
    port: int,
    ready_event: Optional[threading.Event] = None,
    stop_event: Optional[threading.Event] = None,
) -> None:
    """Bind, listen, and serve until ``stop_event`` is set."""
    global _BOUND_PORT
    _BOUND_PORT = port

    server = ThreadingHTTPServer(("", port), MockAPIHandler)

    if ready_event is not None:
        ready_event.set()

    log.info("Listening on http://%s:%d", HOST, port)

    if stop_event is not None:
        def _watch_stop() -> None:
            stop_event.wait()
            server.shutdown()

        threading.Thread(
            target=_watch_stop, name="server-shutdown", daemon=True
        ).start()

    try:
        server.serve_forever(poll_interval=0.2)
    finally:
        server.server_close()
        log.info("Server stopped")


# ============================================================
# Client Helpers
# ============================================================

_session = requests.Session()


def api_url(port: int, path: str) -> str:
    return f"http://{HOST}:{port}{path}"


def get_users(port: int) -> requests.Response:
    return _session.get(api_url(port, "/api/users"), timeout=REQUEST_TIMEOUT)


def get_user(port: int, user_id: int) -> requests.Response:
    return _session.get(
        api_url(port, f"/api/users/{user_id}"), timeout=REQUEST_TIMEOUT
    )


def create_user(port: int, name: str, role: str) -> requests.Response:
    return _session.post(
        api_url(port, "/api/users"),
        json={"name": name, "role": role},
        timeout=REQUEST_TIMEOUT,
    )


def replace_user(port: int, user_id: int, name: str, role: str) -> requests.Response:
    return _session.put(
        api_url(port, f"/api/users/{user_id}"),
        json={"name": name, "role": role},
        timeout=REQUEST_TIMEOUT,
    )


def patch_user_api(port: int, user_id: int, **fields: Any) -> requests.Response:
    return _session.patch(
        api_url(port, f"/api/users/{user_id}"),
        json=fields,
        timeout=REQUEST_TIMEOUT,
    )


def delete_user_api(port: int, user_id: int) -> requests.Response:
    return _session.delete(
        api_url(port, f"/api/users/{user_id}"), timeout=REQUEST_TIMEOUT
    )


def print_response(response: requests.Response) -> None:
    print("Status Code:", response.status_code)
    rid = response.headers.get("X-Request-ID")
    if rid:
        print("X-Request-ID:", rid)
    try:
        print("Response:", response.json())
    except ValueError:
        print("Response:", response.text)


def list_users_as_table(api_port: int) -> None:
    try:
        response = get_users(api_port)
    except requests.RequestException as exc:
        print(f"An error occurred while fetching users: {exc}")
        return

    if response.status_code != 200:
        print(f"Failed to fetch users. Status code: {response.status_code}")
        return

    users = response.json()
    headers = ["ID", "Name", "Role"]
    col_widths = [6, 20, 15]

    row_format = "| " + " | ".join(f"{{:<{w}}}" for w in col_widths) + " |"
    border = "+" + "+".join("-" * (w + 2) for w in col_widths) + "+"

    print(border)
    print(row_format.format(*headers))
    print(border)
    for user in users:
        print(
            row_format.format(
                user.get("id", ""),
                user.get("name", ""),
                user.get("role", ""),
            )
        )
    print(border)


# ============================================================
# Demo / Test Functions
# ============================================================

def test_get_existing_user(port: int) -> None:
    print("\n--- Testing GET /api/users/1 ---")
    print_response(get_user(port, 1))


def test_get_nonexistent_user(port: int) -> None:
    print("\n--- Testing GET /api/users/99 ---")
    print_response(get_user(port, 99))


def test_get_invalid_user_id(port: int) -> None:
    print("\n--- Testing GET /api/users/invalid_id ---")
    print_response(
        _session.get(
            api_url(port, "/api/users/invalid_id"), timeout=REQUEST_TIMEOUT
        )
    )


def test_create_user(port: int) -> None:
    print("\n--- Testing POST /api/users ---")
    print_response(create_user(port, "Charlie", "Developer"))


def test_invalid_user(port: int) -> None:
    print("\n--- Testing POST /api/users with invalid data ---")
    print_response(create_user(port, "", "User"))


def test_invalid_role(port: int) -> None:
    print("\n--- Testing POST /api/users with disallowed role ---")
    print_response(create_user(port, "Frank", "Superuser"))


def test_missing_name(port: int) -> None:
    print("\n--- Testing POST /api/users with missing name ---")
    print_response(
        _session.post(
            api_url(port, "/api/users"),
            json={"role": "User"},
            timeout=REQUEST_TIMEOUT,
        )
    )


def test_missing_role(port: int) -> None:
    print("\n--- Testing POST /api/users with missing role ---")
    print_response(
        _session.post(
            api_url(port, "/api/users"),
            json={"name": "Eve"},
            timeout=REQUEST_TIMEOUT,
        )
    )


def test_invalid_json(port: int) -> None:
    print("\n--- Testing POST /api/users with invalid JSON ---")
    print_response(
        _session.post(
            api_url(port, "/api/users"),
            data="{not valid json",
            headers={"Content-Type": "application/json"},
            timeout=REQUEST_TIMEOUT,
        )
    )


def test_unsupported_media_type(port: int) -> None:
    print("\n--- Testing POST /api/users with wrong Content-Type ---")
    print_response(
        _session.post(
            api_url(port, "/api/users"),
            data="name=Grace&role=User",
            headers={"Content-Type": "application/x-www-form-urlencoded"},
            timeout=REQUEST_TIMEOUT,
        )
    )


def test_replace_user(port: int) -> None:
    print("\n--- Testing PUT /api/users/2 ---")
    print_response(replace_user(port, 2, "Bobby", "Maintainer"))


def test_patch_user(port: int) -> None:
    print("\n--- Testing PATCH /api/users/1 (role only) ---")
    print_response(patch_user_api(port, 1, role="Developer"))


def test_delete_user(port: int) -> None:
    created = create_user(port, "Temporary", "Guest").json()
    uid = created["id"]
    print(f"\n--- Testing DELETE /api/users/{uid} ---")
    print_response(delete_user_api(port, uid))


def test_method_not_allowed(port: int) -> None:
    print("\n--- Testing DELETE /api/users (method not allowed) ---")
    print_response(
        _session.delete(api_url(port, "/api/users"), timeout=REQUEST_TIMEOUT)
    )


def test_head_request(port: int) -> None:
    print("\n--- Testing HEAD /api/users ---")
    response = _session.head(api_url(port, "/api/users"), timeout=REQUEST_TIMEOUT)
    print("Status Code:", response.status_code)
    print("Content-Length:", response.headers.get("Content-Length"))
    print("Body bytes:", len(response.content))


def test_options_request(port: int) -> None:
    print("\n--- Testing OPTIONS /api/users ---")
    response = _session.options(
        api_url(port, "/api/users"), timeout=REQUEST_TIMEOUT
    )
    print("Status Code:", response.status_code)
    print("Allow:", response.headers.get("Allow"))


def test_pagination(port: int) -> None:
    print("\n--- Testing GET /api/users?page=1&per_page=1 ---")
    print_response(
        _session.get(
            api_url(port, "/api/users"),
            params={"page": 1, "per_page": 1},
            timeout=REQUEST_TIMEOUT,
        )
    )


def test_filter_role(port: int) -> None:
    print("\n--- Testing GET /api/users?role=Admin ---")
    print_response(
        _session.get(
            api_url(port, "/api/users"),
            params={"role": "Admin"},
            timeout=REQUEST_TIMEOUT,
        )
    )


def test_search(port: int) -> None:
    print("\n--- Testing GET /api/users?q=ali ---")
    print_response(
        _session.get(
            api_url(port, "/api/users"),
            params={"q": "ali"},
            timeout=REQUEST_TIMEOUT,
        )
    )


def test_health(port: int) -> None:
    print("\n--- Testing GET /health ---")
    print_response(_session.get(api_url(port, "/health"), timeout=REQUEST_TIMEOUT))


def test_stats(port: int) -> None:
    print("\n--- Testing GET /api/stats ---")
    print_response(_session.get(api_url(port, "/api/stats"), timeout=REQUEST_TIMEOUT))


def test_openapi(port: int) -> None:
    print("\n--- Testing GET /openapi.json ---")
    print_response(
        _session.get(api_url(port, "/openapi.json"), timeout=REQUEST_TIMEOUT)
    )


def test_request_id_echo(port: int) -> None:
    print("\n--- Testing X-Request-ID echo ---")
    custom = "test-trace-123"
    response = _session.get(
        api_url(port, "/health"),
        headers={"X-Request-ID": custom},
        timeout=REQUEST_TIMEOUT,
    )
    print("Sent X-Request-ID:", custom)
    print("Status Code:", response.status_code)
    print("Echoed X-Request-ID:", response.headers.get("X-Request-ID"))


# ============================================================
# Automated Assertions
# ============================================================

def assert_status(response: requests.Response, expected: int) -> None:
    assert response.status_code == expected, (
        f"Expected {expected}, got {response.status_code}: {response.text!r}"
    )


def run_assertion_tests(port: int) -> None:
    print("\n--- Running automated API assertions ---")

    # --- GET list (bare-list backward compatibility) ---
    response = get_users(port)
    assert_status(response, 200)
    assert isinstance(response.json(), list)

    # --- GET single (existing) ---
    response = get_user(port, 1)
    assert_status(response, 200)
    assert response.json()["name"] == "Alice"

    # --- GET single (missing) ---
    response = get_user(port, 9999)
    assert_status(response, 404)

    # --- GET single (bad id) ---
    response = _session.get(
        api_url(port, "/api/users/not-an-int"), timeout=REQUEST_TIMEOUT
    )
    assert_status(response, 400)

    # --- POST create ---
    response = create_user(port, "Test User", "Maintainer")
    assert_status(response, 201)
    assert response.json()["name"] == "Test User"
    created_id = response.json()["id"]

    # --- POST validation: empty name ---
    response = create_user(port, "", "Tester")
    assert_status(response, 400)

    # --- POST validation: disallowed role ---
    response = create_user(port, "Nobody", "Wizard")
    assert_status(response, 400)

    # --- POST validation: missing role ---
    response = _session.post(
        api_url(port, "/api/users"),
        json={"name": "Alone"},
        timeout=REQUEST_TIMEOUT,
    )
    assert_status(response, 400)

    # --- POST validation: invalid JSON ---
    response = _session.post(
        api_url(port, "/api/users"),
        data="{broken",
        headers={"Content-Type": "application/json"},
        timeout=REQUEST_TIMEOUT,
    )
    assert_status(response, 400)

    # --- POST validation: wrong Content-Type -> 415 ---
    response = _session.post(
        api_url(port, "/api/users"),
        data="name=x&role=User",
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        timeout=REQUEST_TIMEOUT,
    )
    assert_status(response, 415)

    # --- PUT replace ---
    response = replace_user(port, created_id, "Renamed", "User")
    assert_status(response, 200)
    assert response.json()["name"] == "Renamed"
    assert response.json()["role"] == "User"

    # --- PUT on missing user ---
    response = replace_user(port, 9999, "Ghost", "User")
    assert_status(response, 404)

    # --- PUT with bad id ---
    response = _session.put(
        api_url(port, "/api/users/abc"),
        json={"name": "X", "role": "User"},
        timeout=REQUEST_TIMEOUT,
    )
    assert_status(response, 400)

    # --- PATCH partial ---
    response = patch_user_api(port, created_id, role="Admin")
    assert_status(response, 200)
    assert response.json()["role"] == "Admin"
    assert response.json()["name"] == "Renamed"  # unchanged

    # --- PATCH with no updatable fields ---
    response = _session.patch(
        api_url(port, f"/api/users/{created_id}"),
        json={"id": 99},
        timeout=REQUEST_TIMEOUT,
    )
    assert_status(response, 400)

    # --- PATCH on missing user ---
    response = patch_user_api(port, 9999, role="Admin")
    assert_status(response, 404)

    # --- DELETE ---
    response = delete_user_api(port, created_id)
    assert_status(response, 200)
    assert response.json()["deleted"] is True
    response = get_user(port, created_id)
    assert_status(response, 404)

    # --- DELETE missing ---
    response = delete_user_api(port, 9999)
    assert_status(response, 404)

    # --- Method not allowed (DELETE on collection) ---
    response = _session.delete(
        api_url(port, "/api/users"), timeout=REQUEST_TIMEOUT
    )
    assert_status(response, 405)
    assert "GET" in response.headers.get("Allow", "")
    assert "POST" in response.headers.get("Allow", "")

    # --- HEAD mirrors GET (headers, no body) ---
    response = _session.head(
        api_url(port, "/health"), timeout=REQUEST_TIMEOUT
    )
    assert_status(response, 200)
    assert response.content == b""
    assert int(response.headers.get("Content-Length", "0")) > 0

    response = _session.head(
        api_url(port, "/api/users"), timeout=REQUEST_TIMEOUT
    )
    assert_status(response, 200)
    assert response.content == b""

    # --- OPTIONS advertises HEAD alongside GET/POST ---
    response = _session.options(
        api_url(port, "/api/users"), timeout=REQUEST_TIMEOUT
    )
    assert_status(response, 204)
    allow = response.headers.get("Allow", "")
    assert "GET" in allow and "POST" in allow and "HEAD" in allow

    # --- CORS is present on normal responses ---
    response = get_users(port)
    assert response.headers.get("Access-Control-Allow-Origin") == "*"
    assert "X-Request-ID" in response.headers.get(
        "Access-Control-Expose-Headers", ""
    )

    # --- Pagination envelope ---
    response = _session.get(
        api_url(port, "/api/users"),
        params={"page": 1, "per_page": 1},
        timeout=REQUEST_TIMEOUT,
    )
    assert_status(response, 200)
    body = response.json()
    assert "data" in body and "total" in body and "total_pages" in body
    assert len(body["data"]) <= 1

    # --- Pagination out of range / bad values ---
    response = _session.get(
        api_url(port, "/api/users"),
        params={"page": 0},
        timeout=REQUEST_TIMEOUT,
    )
    assert_status(response, 400)

    for pp in (0, MAX_PAGE_SIZE + 1):
        response = _session.get(
            api_url(port, "/api/users"),
            params={"per_page": pp},
            timeout=REQUEST_TIMEOUT,
        )
        assert_status(response, 400)

    response = _session.get(
        api_url(port, "/api/users"),
        params={"page": "abc"},
        timeout=REQUEST_TIMEOUT,
    )
    assert_status(response, 400)

    # --- Filter by role (case-insensitive) ---
    response = _session.get(
        api_url(port, "/api/users"),
        params={"role": "admin"},
        timeout=REQUEST_TIMEOUT,
    )
    assert_status(response, 200)
    assert all(u["role"] == "Admin" for u in response.json())

    # --- Health ---
    response = _session.get(api_url(port, "/health"), timeout=REQUEST_TIMEOUT)
    assert_status(response, 200)
    assert response.json()["status"] == "ok"
    assert response.json()["version"] == API_VERSION

    # --- Stats ---
    response = _session.get(api_url(port, "/api/stats"), timeout=REQUEST_TIMEOUT)
    assert_status(response, 200)
    assert response.json()["requests_total"] > 0
    assert "user_count" in response.json()
    assert "roles" in response.json()

    # --- OpenAPI: valid version + actual bound port advertised ---
    response = _session.get(
        api_url(port, "/openapi.json"), timeout=REQUEST_TIMEOUT
    )
    assert_status(response, 200)
    spec = response.json()
    assert spec["openapi"].startswith("3.")
    assert spec["servers"][0]["url"].endswith(f":{port}")

    # --- X-Request-ID echo ---
    custom = "abc-echo-xyz"
    response = _session.get(
        api_url(port, "/health"),
        headers={"X-Request-ID": custom},
        timeout=REQUEST_TIMEOUT,
    )
    assert response.headers.get("X-Request-ID") == custom

    # --- X-Request-ID auto-generated ---
    response = get_users(port)
    assert response.headers.get("X-Request-ID")

    print("All automated API assertions passed.")


# ============================================================
# CLI / Entry Point
# ============================================================

def _parse_args(argv: Optional[list[str]] = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Local mock HTTP API simulator.",
    )
    parser.add_argument(
        "--port",
        type=int,
        default=None,
        help=(
            "Port to listen on. Defaults to MOCK_API_PORT, "
            f"then {DEFAULT_PORT}."
        ),
    )
    parser.add_argument(
        "--no-tests",
        action="store_true",
        help="Start the server without running the demo suite.",
    )
    parser.add_argument(
        "-v",
        "--verbose",
        action="store_true",
        help="Enable DEBUG-level logging.",
    )
    return parser.parse_args(argv)


def _resolve_port(cli_port: Optional[int]) -> int:
    if cli_port is not None:
        return cli_port
    env = os.environ.get("MOCK_API_PORT")
    if env:
        try:
            return int(env)
        except ValueError as exc:
            raise SystemExit(
                f"MOCK_API_PORT must be an integer, got {env!r}"
            ) from exc
    return DEFAULT_PORT


def _run_demo(port: int) -> None:
    print("=" * 60)
    print("MOCK API TEST")
    print("=" * 60)

    test_get_existing_user(port)
    test_get_nonexistent_user(port)
    test_get_invalid_user_id(port)

    print("\n--- Testing GET /api/users ---")
    list_users_as_table(port)

    test_create_user(port)
    print("\n--- Users After Creating Charlie ---")
    list_users_as_table(port)

    test_invalid_user(port)
    test_invalid_role(port)
    test_missing_name(port)
    test_missing_role(port)
    test_invalid_json(port)
    test_unsupported_media_type(port)

    test_replace_user(port)
    test_patch_user(port)
    test_delete_user(port)

    test_head_request(port)
    test_options_request(port)
    test_pagination(port)
    test_filter_role(port)
    test_search(port)
    test_health(port)
    test_stats(port)
    test_openapi(port)
    test_request_id_echo(port)
    test_method_not_allowed(port)

    print("\n--- Final User List ---")
    list_users_as_table(port)

    # Reset so the assertion suite runs against a deterministic state.
    print("\n--- Resetting DB + stats before automated assertions ---")
    reset_database()
    _reset_stats()

    run_assertion_tests(port)


def main(argv: Optional[list[str]] = None) -> int:
    args = _parse_args(argv)

    logging.basicConfig(
        level=logging.DEBUG if args.verbose else logging.INFO,
        format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    )

    reset_database()
    _reset_stats()
    port = _resolve_port(args.port)

    ready = threading.Event()
    stop = threading.Event()

    server_thread = threading.Thread(
        target=run_server,
        args=(port, ready, stop),
        name="mock-api-server",
        daemon=True,
    )
    server_thread.start()

    if not ready.wait(timeout=5):
        log.error("Server failed to start within 5 seconds")
        return 1

    if not args.no_tests:
        try:
            _run_demo(port)
        except Exception:
            log.exception("Demo suite crashed")
            stop.set()
            server_thread.join(timeout=5)
            return 2

    log.info(
        "API simulator is running at http://%s:%d (Ctrl+C to stop)",
        HOST,
        port,
    )

    try:
        server_thread.join()
    except KeyboardInterrupt:
        log.info("Shutting down...")
    finally:
        stop.set()
        server_thread.join(timeout=5)

    return 0


if __name__ == "__main__":
    # Check if running in Google Colab / Jupyter environment to bypass CLI parsing
    if "ipykernel" in sys.modules or "google.colab" in sys.modules:
        # Start on dynamically assigned free port
        def get_free_port() -> int:
            s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            s.bind(("", 0))
            p = s.getsockname()[1]
            s.close()
            return p

        port = get_free_port()
        reset_database()
        _reset_stats()
        ready = threading.Event()
        server_thread = threading.Thread(
            target=run_server,
            args=(port, ready),
            name="mock-api-server",
            daemon=True,
        )
        server_thread.start()
        if ready.wait(timeout=5):
            _run_demo(port)
    else:
        sys.exit(main())


MOCK API TEST

--- Testing GET /api/users/1 ---
Status Code: 200
X-Request-ID: 1680aa9cb9864f9796dc63212efcfaf5
Response: {'id': 1, 'name': 'Alice', 'role': 'Admin'}

--- Testing GET /api/users/99 ---
Status Code: 404
X-Request-ID: c3be2e3ff49048d4a53336adf3d9c8c6
Response: {'error': 'Not Found', 'message': 'User with ID 99 not found'}

--- Testing GET /api/users/invalid_id ---
Status Code: 400
X-Request-ID: d6fb864f76044f26964dbc5046acf10e
Response: {'error': 'Bad Request', 'message': 'User ID must be an integer'}

--- Testing GET /api/users ---
+--------+----------------------+-----------------+
| ID     | Name                 | Role            |
+--------+----------------------+-----------------+
| 1      | Alice                | Admin           |
| 2      | Bob                  | User            |
+--------+----------------------+-----------------+

--- Testing POST /api/users ---
Status Code: 201
X-Request-ID: 2b8d865a2eec4634a1fb2f34974eda8e
Response: {'id': 3, 'name': 'Charlie',

In [18]:
"""
Mock API Simulator (v1.4.0)
===========================

A self-contained local HTTP API simulator built on Python's standard
library, plus the external ``requests`` package for the client-side
smoke tests.

Endpoints
---------
GET    /                       Service index (name, version, link list)
GET    /health                 Liveness probe (status, version, uptime)
GET    /api/stats              Aggregate request + data statistics
GET    /openapi.json           OpenAPI 3.0 document for the whole API

GET    /api/users              List users (?page, ?per_page, ?role, ?q)
GET    /api/users/<id>         Fetch a single user
POST   /api/users              Create a user
PUT    /api/users/<id>         Replace a user (full update)
PATCH  /api/users/<id>         Partially update a user
DELETE /api/users/<id>         Delete a user

Every response carries an ``X-Request-ID`` header (echoed when the
client sends one, generated otherwise) for tracing.

``HEAD`` mirrors ``GET`` (headers only), ``OPTIONS`` returns ``204`` with
an ``Allow`` header, and all responses include permissive CORS headers
so browser-based test harnesses can call the mock directly.

Configuration
-------------
By default the server listens on port 4221. Override with ``--port`` or
the ``MOCK_API_PORT`` environment variable.

Usage
-----
    python mock_api.py                 # :4221 + smoke tests
    python mock_api.py --no-tests      # server only
    python mock_api.py --port 8080
    MOCK_API_PORT=9000 python mock_api.py -v

    # In a Jupyter / Colab notebook, just `import mock_api` or run the
    # cell -- a free port is picked automatically and the demo runs.
"""

from __future__ import annotations

import argparse
import copy
import json
import logging
import os
import re
import socket
import sys
import threading
import time
import uuid
from datetime import datetime, timezone
from http.server import BaseHTTPRequestHandler, HTTPServer
from socketserver import ThreadingMixIn
from typing import Any, Callable, NamedTuple, Optional
from urllib.parse import parse_qs, urlsplit

import requests


# ============================================================
# Configuration
# ============================================================

HOST = "localhost"
DEFAULT_PORT = 4221
REQUEST_TIMEOUT = 5
MAX_BODY_BYTES = 1_000_000  # 1 MB cap on incoming request bodies
API_VERSION = "1.4.0"

MAX_NAME_LEN = 100
MAX_ROLE_LEN = 50
ALLOWED_ROLES = {"Admin", "User", "Developer", "Maintainer", "Guest"}

DEFAULT_PAGE_SIZE = 10
MAX_PAGE_SIZE = 100

CORS_ALLOW_ORIGIN = "*"
CORS_ALLOW_METHODS = "GET, HEAD, POST, PUT, PATCH, DELETE, OPTIONS"
CORS_ALLOW_HEADERS = "Content-Type, X-Request-ID"
CORS_EXPOSE_HEADERS = "X-Request-ID, Allow"

log = logging.getLogger("mockapi")

# Set by run_server() so /openapi.json can advertise the real bound port.
_BOUND_PORT: Optional[int] = None


# ============================================================
# Mock Database
# ============================================================

_INITIAL_USERS = [
    {"id": 1, "name": "Alice", "role": "Admin"},
    {"id": 2, "name": "Bob", "role": "User"},
]

# Kept as a public name for backward compatibility; treat as private.
Mock_DB: dict[str, list[dict[str, Any]]] = {
    "users": [dict(u) for u in _INITIAL_USERS],
}

# RLock keeps us safe even if a helper calls another lock-acquiring helper.
_db_lock = threading.RLock()


def reset_database() -> None:
    with _db_lock:
        Mock_DB["users"].clear()
        Mock_DB["users"].extend(dict(u) for u in _INITIAL_USERS)


def next_user_id() -> int:
    with _db_lock:
        if not Mock_DB["users"]:
            return 1
        return max(u["id"] for u in Mock_DB["users"]) + 1


def get_user_by_id(user_id: int) -> Optional[dict[str, Any]]:
    """Return a *copy* of the user, or ``None`` if not found."""
    with _db_lock:
        for user in Mock_DB["users"]:
            if user["id"] == user_id:
                return dict(user)
    return None


def list_users() -> list[dict[str, Any]]:
    with _db_lock:
        return [dict(u) for u in Mock_DB["users"]]


def add_user(name: str, role: str) -> dict[str, Any]:
    with _db_lock:
        new_user = {"id": next_user_id(), "name": name, "role": role}
        Mock_DB["users"].append(new_user)
        return dict(new_user)


def update_user(user_id: int, name: str, role: str) -> Optional[dict[str, Any]]:
    with _db_lock:
        for user in Mock_DB["users"]:
            if user["id"] == user_id:
                user["name"] = name
                user["role"] = role
                return dict(user)
    return None


def patch_user(
    user_id: int, *, name: Optional[str] = None, role: Optional[str] = None
) -> Optional[dict[str, Any]]:
    """Partial update. Only supplied fields are changed.

    Both ``name`` and ``role`` are applied *independently* -- passing
    only ``role`` must not require ``name`` to be present.
    """
    with _db_lock:
        for user in Mock_DB["users"]:
            if user["id"] == user_id:
                if name is not None:
                    user["name"] = name
                if role is not None:
                    user["role"] = role
                return dict(user)
    return None


def delete_user(user_id: int) -> bool:
    with _db_lock:
        for i, user in enumerate(Mock_DB["users"]):
            if user["id"] == user_id:
                Mock_DB["users"].pop(i)
                return True
    return False


# ============================================================
# Request Statistics
# ============================================================

_stats_lock = threading.Lock()
_stats: dict[str, Any] = {
    "started_at": datetime.now(timezone.utc).isoformat(),
    "start_perf": time.monotonic(),
    "requests_total": 0,
    "by_method": {},
    "by_path": {},
}


def _bump_stats(method: str, path: str) -> None:
    with _stats_lock:
        _stats["requests_total"] += 1
        _stats["by_method"][method] = _stats["by_method"].get(method, 0) + 1
        _stats["by_path"][path] = _stats["by_path"].get(path, 0) + 1


def _snapshot_stats() -> dict[str, Any]:
    with _stats_lock:
        return {
            "requests_total": _stats["requests_total"],
            "by_method": dict(_stats["by_method"]),
            "by_path": dict(_stats["by_path"]),
            "started_at": _stats["started_at"],
            "uptime_seconds": round(time.monotonic() - _stats["start_perf"], 3),
        }


def _reset_stats() -> None:
    with _stats_lock:
        _stats["started_at"] = datetime.now(timezone.utc).isoformat()
        _stats["start_perf"] = time.monotonic()
        _stats["requests_total"] = 0
        _stats["by_method"] = {}
        _stats["by_path"] = {}


# ============================================================
# Validation
# ============================================================

class UnsupportedMediaType(ValueError):
    """Raised when the request's Content-Type is not application/json."""

    status = 415


class ValidatedUser(NamedTuple):
    """Result of validate_user_payload().

    On success ``error`` is ``None``. ``name``/``role`` are ``str`` when
    ``require_all=True``; for PATCH (``require_all=False``) they are
    ``None`` when the field was not supplied.
    """

    name: Optional[str]
    role: Optional[str]
    error: Optional[str]


def validate_user_payload(
    payload: dict[str, Any], require_all: bool = True
) -> ValidatedUser:
    """Validate a user payload.

    When ``require_all`` is False (PATCH semantics) only present fields
    are validated and missing ones come back as ``None`` without error.
    """
    name = payload.get("name")
    role = payload.get("role")

    if require_all:
        if not isinstance(name, str) or not name.strip() or len(name) > MAX_NAME_LEN:
            return ValidatedUser(
                None,
                None,
                f"'name' is required and must be a non-empty "
                f"string of at most {MAX_NAME_LEN} characters",
            )
        if not isinstance(role, str) or not role.strip() or len(role) > MAX_ROLE_LEN:
            return ValidatedUser(
                None,
                None,
                f"'role' is required and must be a non-empty "
                f"string of at most {MAX_ROLE_LEN} characters",
            )
        if role.strip() not in ALLOWED_ROLES:
            return ValidatedUser(
                None,
                None,
                f"'role' must be one of {sorted(ALLOWED_ROLES)}",
            )
        return ValidatedUser(name.strip(), role.strip(), None)

    clean_name: Optional[str] = None
    clean_role: Optional[str] = None
    if name is not None:
        if not isinstance(name, str) or not name.strip() or len(name) > MAX_NAME_LEN:
            return ValidatedUser(
                None,
                None,
                f"'name' must be a non-empty string of at most "
                f"{MAX_NAME_LEN} characters",
            )
        clean_name = name.strip()
    if role is not None:
        if not isinstance(role, str) or not role.strip() or len(role) > MAX_ROLE_LEN:
            return ValidatedUser(
                None,
                None,
                f"'role' must be a non-empty string of at most "
                f"{MAX_ROLE_LEN} characters",
            )
        if role.strip() not in ALLOWED_ROLES:
            return ValidatedUser(
                None, None, f"'role' must be one of {sorted(ALLOWED_ROLES)}"
            )
        clean_role = role.strip()
    return ValidatedUser(clean_name, clean_role, None)


# ============================================================
# Router
# ============================================================

class Route:
    """A single (method, regex) -> handler binding."""

    __slots__ = ("method", "pattern", "handler")

    def __init__(
        self, method: str, pattern: str, handler: Callable[..., None]
    ) -> None:
        self.method = method
        self.pattern = re.compile(pattern)
        self.handler = handler


ROUTES: list[Route] = []


def route(
    method: str, pattern: str
) -> Callable[[Callable[..., None]], Callable[..., None]]:
    """Register a handler for ``method`` matching ``pattern`` (anchored)."""

    def decorator(fn: Callable[..., None]) -> Callable[..., None]:
        ROUTES.append(Route(method, pattern, fn))
        return fn

    return decorator


def _allowed_methods_for(path: str) -> list[str]:
    methods = {r.method for r in ROUTES if r.pattern.match(path)}
    if "GET" in methods:
        methods.add("HEAD")
    return sorted(methods)


def _parse_int(qs: dict[str, list[str]], key: str, default: int) -> int:
    values = qs.get(key)
    if not values:
        return default
    try:
        return int(values[0])
    except ValueError as exc:
        raise ValueError(f"'{key}' must be an integer") from exc


# ============================================================
# API Request Handler
# ============================================================

class MockAPIHandler(BaseHTTPRequestHandler):
    """Routes the mock API. Each method must write exactly one response."""

    protocol_version = "HTTP/1.0"
    server_version = f"MockAPI/{API_VERSION}"

    # ---------------- response helpers ----------------

    def _send_cors_headers(self) -> None:
        self.send_header("Access-Control-Allow-Origin", CORS_ALLOW_ORIGIN)
        self.send_header("Access-Control-Allow-Methods", CORS_ALLOW_METHODS)
        self.send_header("Access-Control-Allow-Headers", CORS_ALLOW_HEADERS)
        self.send_header("Access-Control-Expose-Headers", CORS_EXPOSE_HEADERS)

    def _send_common_headers(self) -> None:
        self.send_header("Connection", "close")
        rid = getattr(self, "_request_id", None)
        if rid:
            self.send_header("X-Request-ID", rid)
        self._send_cors_headers()

    def _send_json(
        self,
        payload: Any,
        status_code: int = 200,
        extra_headers: Optional[dict[str, str]] = None,
    ) -> None:
        body = json.dumps(payload).encode("utf-8")
        self.send_response(status_code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self._send_common_headers()
        for k, v in (extra_headers or {}).items():
            self.send_header(k, v)
        self.end_headers()
        if self.command != "HEAD":
            try:
                self.wfile.write(body)
            except (BrokenPipeError, ConnectionResetError):
                pass
        self.close_connection = True

    def _send_no_content(
        self,
        status_code: int = 204,
        extra_headers: Optional[dict[str, str]] = None,
    ) -> None:
        self.send_response(status_code)
        self.send_header("Content-Length", "0")
        self._send_common_headers()
        for k, v in (extra_headers or {}).items():
            self.send_header(k, v)
        self.end_headers()
        self.close_connection = True

    def _send_error_json(
        self, status: int, error: str, message: str
    ) -> None:
        self._send_json({"error": error, "message": message}, status)

    def _method_not_allowed(self, *allowed: str) -> None:
        self._send_json(
            {
                "error": "Method Not Allowed",
                "message": f"Allowed methods: {', '.join(allowed)}",
            },
            405,
            extra_headers={"Allow": ", ".join(allowed)},
        )

    # ---------------- request parsing ----------------

    def _get_path(self) -> str:
        return urlsplit(self.path).path

    def _get_query(self) -> dict[str, list[str]]:
        return parse_qs(urlsplit(self.path).query, keep_blank_values=True)

    def _ensure_request_id(self) -> str:
        rid = self.headers.get("X-Request-ID") or uuid.uuid4().hex
        self._request_id = rid
        return rid

    def _read_json_object(self) -> dict[str, Any]:
        ctype = self.headers.get("Content-Type", "")
        if ctype and "application/json" not in ctype.lower():
            raise UnsupportedMediaType("Content-Type must be application/json")

        raw_length = self.headers.get("Content-Length")
        if raw_length is None:
            raise ValueError("Content-Length header is required")
        try:
            length = int(raw_length)
        except ValueError as exc:
            raise ValueError("Content-Length must be an integer") from exc
        if length < 0:
            raise ValueError("Content-Length must be non-negative")
        if length > MAX_BODY_BYTES:
            raise ValueError(
                f"Request body exceeds limit of {MAX_BODY_BYTES} bytes"
            )

        raw = self.rfile.read(length)
        try:
            parsed = json.loads(raw)
        except (UnicodeDecodeError, json.JSONDecodeError) as exc:
            raise ValueError(f"Invalid JSON: {exc}") from exc
        if not isinstance(parsed, dict):
            raise ValueError("Request body must be a JSON object")
        return parsed

    # ---------------- dispatch ----------------

    def _dispatch(self, method: str) -> None:
        path = self._get_path()
        query = self._get_query()
        request_id = self._ensure_request_id()
        _bump_stats(method, path)
        log.debug("[%s] %s %s", request_id, method, path)

        try:
            if method == "OPTIONS":
                allowed = _allowed_methods_for(path)
                if not allowed:
                    self._send_error_json(404, "Not Found", f"No route for {path}")
                    return
                self._send_no_content(204, extra_headers={"Allow": ", ".join(allowed)})
                return

            lookup = "GET" if method == "HEAD" else method
            for r in ROUTES:
                if r.method != lookup:
                    continue
                m = r.pattern.match(path)
                if m:
                    r.handler(self, m, query)
                    return

            allowed = _allowed_methods_for(path)
            if allowed:
                self._method_not_allowed(*allowed)
            else:
                self._send_error_json(404, "Not Found", f"No route for {path}")
        except UnsupportedMediaType as exc:
            self._send_error_json(415, "Unsupported Media Type", str(exc))
        except ValueError as exc:
            self._send_error_json(400, "Bad Request", str(exc))
        except Exception:
            log.exception("[%s] Unhandled error", request_id)
            self._send_error_json(
                500, "Internal Server Error", "Unexpected server error"
            )

    def do_GET(self) -> None:  # noqa: N802
        self._dispatch("GET")

    def do_HEAD(self) -> None:  # noqa: N802
        self._dispatch("HEAD")

    def do_POST(self) -> None:  # noqa: N802
        self._dispatch("POST")

    def do_PUT(self) -> None:  # noqa: N802
        self._dispatch("PUT")

    def do_PATCH(self) -> None:  # noqa: N802
        self._dispatch("PATCH")

    def do_DELETE(self) -> None:  # noqa: N802
        self._dispatch("DELETE")

    def do_OPTIONS(self) -> None:  # noqa: N802
        self._dispatch("OPTIONS")

    def log_message(self, fmt: str, *args: Any) -> None:
        log.debug("%s - %s", self.address_string(), fmt % args)


# ============================================================
# Route Handlers
# ============================================================

@route("GET", r"^/api/users/?$")
def handle_list_users(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    users = list_users()

    role = (qs.get("role") or [None])[0]
    search = (qs.get("q") or [None])[0]
    if role:
        users = [u for u in users if u["role"].lower() == role.lower()]
    if search:
        needle = search.lower()
        users = [u for u in users if needle in u["name"].lower()]

    has_paging = any(k in qs for k in ("page", "per_page"))
    if has_paging:
        page = _parse_int(qs, "page", 1)
        per_page = _parse_int(qs, "per_page", DEFAULT_PAGE_SIZE)
        if page < 1:
            raise ValueError("'page' must be >= 1")
        if per_page < 1 or per_page > MAX_PAGE_SIZE:
            raise ValueError(
                f"'per_page' must be between 1 and {MAX_PAGE_SIZE}"
            )
        total = len(users)
        total_pages = (total + per_page - 1) // per_page if total else 0
        start = (page - 1) * per_page
        h._send_json(
            {
                "data": users[start:start + per_page],
                "page": page,
                "per_page": per_page,
                "total": total,
                "total_pages": total_pages,
            },
            200,
        )
    else:
        h._send_json(users, 200)


@route("GET", r"^/api/users/(\d+)/?$")
def handle_get_user(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    user_id = int(m.group(1))
    user = get_user_by_id(user_id)
    if user is None:
        h._send_error_json(404, "Not Found", f"User with ID {user_id} not found")
        return
    h._send_json(user, 200)


@route("GET", r"^/api/users/([^/]+)/?$")
def handle_get_user_bad_id(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    h._send_error_json(400, "Bad Request", "User ID must be an integer")


@route("POST", r"^/api/users/?$")
def handle_create_user(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    payload = h._read_json_object()
    name, role, err = validate_user_payload(payload, require_all=True)
    if err:
        h._send_error_json(400, "Invalid input", err)
        return
    assert name is not None and role is not None
    h._send_json(add_user(name, role), 201)


@route("PUT", r"^/api/users/(\d+)/?$")
def handle_replace_user(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    user_id = int(m.group(1))
    if get_user_by_id(user_id) is None:
        h._send_error_json(404, "Not Found", f"User with ID {user_id} not found")
        return
    payload = h._read_json_object()
    name, role, err = validate_user_payload(payload, require_all=True)
    if err:
        h._send_error_json(400, "Invalid input", err)
        return
    assert name is not None and role is not None
    h._send_json(update_user(user_id, name, role), 200)


@route("PUT", r"^/api/users/([^/]+)/?$")
def handle_replace_user_bad_id(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    h._send_error_json(400, "Bad Request", "User ID must be an integer")


@route("PATCH", r"^/api/users/(\d+)/?$")
def handle_patch_user(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    user_id = int(m.group(1))
    if get_user_by_id(user_id) is None:
        h._send_error_json(404, "Not Found", f"User with ID {user_id} not found")
        return
    payload = h._read_json_object()
    name, role, err = validate_user_payload(payload, require_all=False)
    if err:
        h._send_error_json(400, "Invalid input", err)
        return
    if name is None and role is None:
        h._send_error_json(
            400,
            "Invalid input",
            "No valid fields to update; supply 'name' and/or 'role'",
        )
        return
    h._send_json(patch_user(user_id, name=name, role=role), 200)


@route("PATCH", r"^/api/users/([^/]+)/?$")
def handle_patch_user_bad_id(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    h._send_error_json(400, "Bad Request", "User ID must be an integer")


@route("DELETE", r"^/api/users/(\d+)/?$")
def handle_delete_user(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    user_id = int(m.group(1))
    if delete_user(user_id):
        h._send_json({"deleted": True, "id": user_id}, 200)
    else:
        h._send_error_json(404, "Not Found", f"User with ID {user_id} not found")


@route("DELETE", r"^/api/users/([^/]+)/?$")
def handle_delete_user_bad_id(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    h._send_error_json(400, "Bad Request", "User ID must be an integer")


@route("GET", r"^/health/?$")
def handle_health(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    snapshot = _snapshot_stats()
    h._send_json(
        {
            "status": "ok",
            "version": API_VERSION,
            "uptime_seconds": snapshot["uptime_seconds"],
        },
        200,
    )


@route("GET", r"^/api/stats/?$")
def handle_stats(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    snapshot = _snapshot_stats()
    users = list_users()
    roles: dict[str, int] = {}
    for u in users:
        roles[u["role"]] = roles.get(u["role"], 0) + 1
    snapshot["user_count"] = len(users)
    snapshot["roles"] = roles
    h._send_json(snapshot, 200)


@route("GET", r"^/openapi\.json/?$")
def handle_openapi(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    spec = copy.deepcopy(OPENAPI_SPEC)
    if _BOUND_PORT is not None:
        spec["servers"] = [{"url": f"http://{HOST}:{_BOUND_PORT}"}]
    h._send_json(spec, 200)


@route("GET", r"^/?$")
def handle_root(
    h: MockAPIHandler, m: re.Match[str], qs: dict[str, list[str]]
) -> None:
    h._send_json(
        {
            "name": "Mock API",
            "version": API_VERSION,
            "endpoints": [
                "GET /api/users",
                "GET /api/users/<id>",
                "POST /api/users",
                "PUT /api/users/<id>",
                "PATCH /api/users/<id>",
                "DELETE /api/users/<id>",
                "GET /health",
                "GET /api/stats",
                "GET /openapi.json",
            ],
        },
        200,
    )


# ============================================================
# OpenAPI 3.0 Document
# ============================================================

OPENAPI_SPEC: dict[str, Any] = {
    "openapi": "3.0.3",
    "info": {
        "title": "Mock API",
        "version": API_VERSION,
        "description": "A self-contained local mock HTTP API simulator.",
    },
    "servers": [{"url": f"http://{HOST}:{DEFAULT_PORT}"}],
    "paths": {
        "/": {
            "get": {
                "summary": "Service index",
                "responses": {"200": {"description": "API metadata"}},
            }
        },
        "/api/users": {
            "get": {
                "summary": "List users",
                "description": (
                    "Returns all users as a bare list. Add `page` or "
                    "`per_page` to get a paginated envelope; filter with "
                    "`role` and search with `q`."
                ),
                "parameters": [
                    {
                        "name": "page",
                        "in": "query",
                        "schema": {"type": "integer", "minimum": 1},
                    },
                    {
                        "name": "per_page",
                        "in": "query",
                        "schema": {
                            "type": "integer",
                            "minimum": 1,
                            "maximum": MAX_PAGE_SIZE,
                        },
                    },
                    {"name": "role", "in": "query", "schema": {"type": "string"}},
                    {"name": "q", "in": "query", "schema": {"type": "string"}},
                ],
                "responses": {
                    "200": {
                        "description": "User list or paginated envelope",
                        "content": {"application/json": {"schema": {}}},
                    }
                },
            },
            "post": {
                "summary": "Create a user",
                "requestBody": {
                    "required": True,
                    "content": {
                        "application/json": {
                            "schema": {"$ref": "#/components/schemas/UserCreate"}
                        }
                    },
                },
                "responses": {
                    "201": {"description": "Created user"},
                    "400": {"description": "Invalid input"},
                    "415": {"description": "Unsupported Media Type"},
                },
            },
        },
        "/api/users/{id}": {
            "get": {
                "summary": "Fetch a user",
                "responses": {
                    "200": {"description": "The user"},
                    "404": {"description": "Not found"},
                },
            },
            "put": {
                "summary": "Replace a user",
                "responses": {
                    "200": {"description": "Updated user"},
                    "400": {"description": "Invalid input"},
                    "404": {"description": "Not found"},
                },
            },
            "patch": {
                "summary": "Partially update a user",
                "responses": {
                    "200": {"description": "Updated user"},
                    "400": {"description": "Invalid input"},
                    "404": {"description": "Not found"},
                },
            },
            "delete": {
                "summary": "Delete a user",
                "responses": {
                    "200": {"description": "Deleted"},
                    "404": {"description": "Not found"},
                },
            },
        },
        "/health": {
            "get": {
                "summary": "Liveness probe",
                "responses": {"200": {"description": "Service health"}},
            }
        },
        "/api/stats": {
            "get": {
                "summary": "Request and data statistics",
                "responses": {"200": {"description": "Statistics snapshot"}},
            }
        },
        "/openapi.json": {
            "get": {
                "summary": "OpenAPI document",
                "responses": {"200": {"description": "OpenAPI 3.0 JSON"}},
            }
        },
    },
    "components": {
        "schemas": {
            "UserCreate": {
                "type": "object",
                "required": ["name", "role"],
                "properties": {
                    "name": {"type": "string", "maxLength": MAX_NAME_LEN},
                    "role": {"type": "string", "enum": sorted(ALLOWED_ROLES)},
                },
            },
            "User": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer"},
                    "name": {"type": "string"},
                    "role": {"type": "string"},
                },
            },
        }
    },
}


# ============================================================
# HTTP Server
# ============================================================

class ThreadingHTTPServer(ThreadingMixIn, HTTPServer):
    """HTTPServer that handles each request on its own daemon thread."""

    allow_reuse_address = True
    daemon_threads = True

    def handle_error(
        self, request: socket.socket, client_address: Any
    ) -> None:
        exc = sys.exc_info()[1]
        if isinstance(exc, (BrokenPipeError, ConnectionResetError)):
            return
        super().handle_error(request, client_address)


def run_server(
    port: int,
    ready_event: Optional[threading.Event] = None,
    stop_event: Optional[threading.Event] = None,
) -> None:
    """Bind, listen, and serve until ``stop_event`` is set."""
    global _BOUND_PORT
    _BOUND_PORT = port

    server = ThreadingHTTPServer(("", port), MockAPIHandler)

    if ready_event is not None:
        ready_event.set()

    log.info("Listening on http://%s:%d", HOST, port)

    if stop_event is not None:
        def _watch_stop() -> None:
            stop_event.wait()
            server.shutdown()

        threading.Thread(
            target=_watch_stop, name="server-shutdown", daemon=True
        ).start()

    try:
        server.serve_forever(poll_interval=0.2)
    finally:
        server.server_close()
        log.info("Server stopped")


# ============================================================
# Client Helpers
# ============================================================

_session = requests.Session()


def api_url(port: int, path: str) -> str:
    return f"http://{HOST}:{port}{path}"


def get_users(port: int) -> requests.Response:
    return _session.get(api_url(port, "/api/users"), timeout=REQUEST_TIMEOUT)


def get_user(port: int, user_id: int) -> requests.Response:
    return _session.get(
        api_url(port, f"/api/users/{user_id}"), timeout=REQUEST_TIMEOUT
    )


def create_user(port: int, name: str, role: str) -> requests.Response:
    return _session.post(
        api_url(port, "/api/users"),
        json={"name": name, "role": role},
        timeout=REQUEST_TIMEOUT,
    )


def replace_user(port: int, user_id: int, name: str, role: str) -> requests.Response:
    return _session.put(
        api_url(port, f"/api/users/{user_id}"),
        json={"name": name, "role": role},
        timeout=REQUEST_TIMEOUT,
    )


def patch_user_api(port: int, user_id: int, **fields: Any) -> requests.Response:
    return _session.patch(
        api_url(port, f"/api/users/{user_id}"),
        json=fields,
        timeout=REQUEST_TIMEOUT,
    )


def delete_user_api(port: int, user_id: int) -> requests.Response:
    return _session.delete(
        api_url(port, f"/api/users/{user_id}"), timeout=REQUEST_TIMEOUT
    )


def print_response(response: requests.Response) -> None:
    print("Status Code:", response.status_code)
    rid = response.headers.get("X-Request-ID")
    if rid:
        print("X-Request-ID:", rid)
    try:
        print("Response:", response.json())
    except ValueError:
        print("Response:", response.text)


def list_users_as_table(api_port: int) -> None:
    try:
        response = get_users(api_port)
    except requests.RequestException as exc:
        print(f"An error occurred while fetching users: {exc}")
        return

    if response.status_code != 200:
        print(f"Failed to fetch users. Status code: {response.status_code}")
        return

    users = response.json()
    headers = ["ID", "Name", "Role"]
    col_widths = [6, 20, 15]

    row_format = "| " + " | ".join(f"{{:<{w}}}" for w in col_widths) + " |"
    border = "+" + "+".join("-" * (w + 2) for w in col_widths) + "+"

    print(border)
    print(row_format.format(*headers))
    print(border)
    for user in users:
        print(
            row_format.format(
                str(user.get("id", "")),
                user.get("name", ""),
                user.get("role", ""),
            )
        )
    print(border)


# ============================================================
# Demo / Test Functions
# ============================================================

def test_get_existing_user(port: int) -> None:
    print("\n--- Testing GET /api/users/1 ---")
    print_response(get_user(port, 1))


def test_get_nonexistent_user(port: int) -> None:
    print("\n--- Testing GET /api/users/99 ---")
    print_response(get_user(port, 99))


def test_get_invalid_user_id(port: int) -> None:
    print("\n--- Testing GET /api/users/invalid_id ---")
    print_response(
        _session.get(
            api_url(port, "/api/users/invalid_id"), timeout=REQUEST_TIMEOUT
        )
    )


def test_create_user(port: int) -> None:
    print("\n--- Testing POST /api/users ---")
    print_response(create_user(port, "Charlie", "Developer"))


def test_invalid_user(port: int) -> None:
    print("\n--- Testing POST /api/users with invalid data ---")
    print_response(create_user(port, "", "User"))


def test_invalid_role(port: int) -> None:
    print("\n--- Testing POST /api/users with disallowed role ---")
    print_response(create_user(port, "Frank", "Superuser"))


def test_missing_name(port: int) -> None:
    print("\n--- Testing POST /api/users with missing name ---")
    print_response(
        _session.post(
            api_url(port, "/api/users"),
            json={"role": "User"},
            timeout=REQUEST_TIMEOUT,
        )
    )


def test_missing_role(port: int) -> None:
    print("\n--- Testing POST /api/users with missing role ---")
    print_response(
        _session.post(
            api_url(port, "/api/users"),
            json={"name": "Eve"},
            timeout=REQUEST_TIMEOUT,
        )
    )


def test_invalid_json(port: int) -> None:
    print("\n--- Testing POST /api/users with invalid JSON ---")
    print_response(
        _session.post(
            api_url(port, "/api/users"),
            data="{not valid json",
            headers={"Content-Type": "application/json"},
            timeout=REQUEST_TIMEOUT,
        )
    )


def test_unsupported_media_type(port: int) -> None:
    print("\n--- Testing POST /api/users with wrong Content-Type ---")
    print_response(
        _session.post(
            api_url(port, "/api/users"),
            data="name=Grace&role=User",
            headers={"Content-Type": "application/x-www-form-urlencoded"},
            timeout=REQUEST_TIMEOUT,
        )
    )


def test_replace_user(port: int) -> None:
    print("\n--- Testing PUT /api/users/2 ---")
    print_response(replace_user(port, 2, "Bobby", "Maintainer"))


def test_patch_user(port: int) -> None:
    print("\n--- Testing PATCH /api/users/1 (role only) ---")
    print_response(patch_user_api(port, 1, role="Developer"))


def test_delete_user(port: int) -> None:
    created = create_user(port, "Temporary", "Guest").json()
    uid = created["id"]
    print(f"\n--- Testing DELETE /api/users/{uid} ---")
    print_response(delete_user_api(port, uid))


def test_method_not_allowed(port: int) -> None:
    print("\n--- Testing DELETE /api/users (method not allowed) ---")
    print_response(
        _session.delete(api_url(port, "/api/users"), timeout=REQUEST_TIMEOUT)
    )


def test_head_request(port: int) -> None:
    print("\n--- Testing HEAD /api/users ---")
    response = _session.head(api_url(port, "/api/users"), timeout=REQUEST_TIMEOUT)
    print("Status Code:", response.status_code)
    print("Content-Length:", response.headers.get("Content-Length"))
    print("Body bytes:", len(response.content))


def test_options_request(port: int) -> None:
    print("\n--- Testing OPTIONS /api/users ---")
    response = _session.options(
        api_url(port, "/api/users"), timeout=REQUEST_TIMEOUT
    )
    print("Status Code:", response.status_code)
    print("Allow:", response.headers.get("Allow"))


def test_pagination(port: int) -> None:
    print("\n--- Testing GET /api/users?page=1&per_page=1 ---")
    print_response(
        _session.get(
            api_url(port, "/api/users"),
            params={"page": 1, "per_page": 1},
            timeout=REQUEST_TIMEOUT,
        )
    )


def test_filter_role(port: int) -> None:
    print("\n--- Testing GET /api/users?role=Admin ---")
    print_response(
        _session.get(
            api_url(port, "/api/users"),
            params={"role": "Admin"},
            timeout=REQUEST_TIMEOUT,
        )
    )


def test_search(port: int) -> None:
    print("\n--- Testing GET /api/users?q=ali ---")
    print_response(
        _session.get(
            api_url(port, "/api/users"),
            params={"q": "ali"},
            timeout=REQUEST_TIMEOUT,
        )
    )


def test_health(port: int) -> None:
    print("\n--- Testing GET /health ---")
    print_response(_session.get(api_url(port, "/health"), timeout=REQUEST_TIMEOUT))


def test_stats(port: int) -> None:
    print("\n--- Testing GET /api/stats ---")
    print_response(_session.get(api_url(port, "/api/stats"), timeout=REQUEST_TIMEOUT))


def test_openapi(port: int) -> None:
    print("\n--- Testing GET /openapi.json ---")
    print_response(
        _session.get(api_url(port, "/openapi.json"), timeout=REQUEST_TIMEOUT)
    )


def test_request_id_echo(port: int) -> None:
    print("\n--- Testing X-Request-ID echo ---")
    custom = "test-trace-123"
    response = _session.get(
        api_url(port, "/health"),
        headers={"X-Request-ID": custom},
        timeout=REQUEST_TIMEOUT,
    )
    print("Sent X-Request-ID:", custom)
    print("Status Code:", response.status_code)
    print("Echoed X-Request-ID:", response.headers.get("X-Request-ID"))


# ============================================================
# Automated Assertions
# ============================================================

def assert_status(response: requests.Response, expected: int) -> None:
    assert response.status_code == expected, (
        f"Expected {expected}, got {response.status_code}: {response.text!r}"
    )


def run_assertion_tests(port: int) -> None:
    print("\n--- Running automated API assertions ---")

    response = get_users(port)
    assert_status(response, 200)
    assert isinstance(response.json(), list)

    response = get_user(port, 1)
    assert_status(response, 200)
    assert response.json()["name"] == "Alice"

    response = get_user(port, 9999)
    assert_status(response, 404)

    response = _session.get(
        api_url(port, "/api/users/not-an-int"), timeout=REQUEST_TIMEOUT
    )
    assert_status(response, 400)

    response = create_user(port, "Test User", "Maintainer")
    assert_status(response, 201)
    assert response.json()["name"] == "Test User"
    created_id = response.json()["id"]

    assert_status(create_user(port, "", "Tester"), 400)
    assert_status(create_user(port, "Nobody", "Wizard"), 400)

    response = _session.post(
        api_url(port, "/api/users"),
        json={"name": "Alone"},
        timeout=REQUEST_TIMEOUT,
    )
    assert_status(response, 400)

    response = _session.post(
        api_url(port, "/api/users"),
        data="{broken",
        headers={"Content-Type": "application/json"},
        timeout=REQUEST_TIMEOUT,
    )
    assert_status(response, 400)

    response = _session.post(
        api_url(port, "/api/users"),
        data="name=x&role=User",
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        timeout=REQUEST_TIMEOUT,
    )
    assert_status(response, 415)

    response = replace_user(port, created_id, "Renamed", "User")
    assert_status(response, 200)
    assert response.json()["name"] == "Renamed"
    assert response.json()["role"] == "User"

    assert_status(replace_user(port, 9999, "Ghost", "User"), 404)

    response = _session.put(
        api_url(port, "/api/users/abc"),
        json={"name": "X", "role": "User"},
        timeout=REQUEST_TIMEOUT,
    )
    assert_status(response, 400)

    response = patch_user_api(port, created_id, role="Admin")
    assert_status(response, 200)
    assert response.json()["role"] == "Admin"
    assert response.json()["name"] == "Renamed"

    response = patch_user_api(port, created_id, name="Renamed2")
    assert_status(response, 200)
    assert response.json()["name"] == "Renamed2"
    assert response.json()["role"] == "Admin"

    response = _session.patch(
        api_url(port, f"/api/users/{created_id}"),
        json={"id": 99},
        timeout=REQUEST_TIMEOUT,
    )
    assert_status(response, 400)

    assert_status(patch_user_api(port, 9999, role="Admin"), 404)

    response = delete_user_api(port, created_id)
    assert_status(response, 200)
    assert response.json()["deleted"] is True
    assert_status(get_user(port, created_id), 404)
    assert_status(delete_user_api(port, 9999), 404)

    response = _session.delete(
        api_url(port, "/api/users"), timeout=REQUEST_TIMEOUT
    )
    assert_status(response, 405)
    assert "GET" in response.headers.get("Allow", "")
    assert "POST" in response.headers.get("Allow", "")

    response = _session.head(api_url(port, "/health"), timeout=REQUEST_TIMEOUT)
    assert_status(response, 200)
    assert response.content == b""
    assert int(response.headers.get("Content-Length", "0")) > 0

    response = _session.head(api_url(port, "/api/users"), timeout=REQUEST_TIMEOUT)
    assert_status(response, 200)
    assert response.content == b""

    response = _session.options(
        api_url(port, "/api/users"), timeout=REQUEST_TIMEOUT
    )
    assert_status(response, 204)
    allow = response.headers.get("Allow", "")
    assert "GET" in allow and "POST" in allow and "HEAD" in allow

    response = get_users(port)
    assert response.headers.get("Access-Control-Allow-Origin") == "*"
    assert "X-Request-ID" in response.headers.get(
        "Access-Control-Expose-Headers", ""
    )

    response = _session.get(
        api_url(port, "/api/users"),
        params={"page": 1, "per_page": 1},
        timeout=REQUEST_TIMEOUT,
    )
    assert_status(response, 200)
    body = response.json()
    assert {"data", "total", "total_pages"} <= body.keys()
    assert len(body["data"]) <= 1

    response = _session.get(
        api_url(port, "/api/users"),
        params={"page": 0},
        timeout=REQUEST_TIMEOUT,
    )
    assert_status(response, 400)

    for pp in (0, MAX_PAGE_SIZE + 1):
        response = _session.get(
            api_url(port, "/api/users"),
            params={"per_page": pp},
            timeout=REQUEST_TIMEOUT,
        )
        assert_status(response, 400)

    response = _session.get(
        api_url(port, "/api/users"),
        params={"page": "abc"},
        timeout=REQUEST_TIMEOUT,
    )
    assert_status(response, 400)

    response = _session.get(
        api_url(port, "/api/users"),
        params={"role": "admin"},
        timeout=REQUEST_TIMEOUT,
    )
    assert_status(response, 200)
    assert all(u["role"] == "Admin" for u in response.json())

    response = _session.get(api_url(port, "/health"), timeout=REQUEST_TIMEOUT)
    assert_status(response, 200)
    assert response.json()["status"] == "ok"
    assert response.json()["version"] == API_VERSION

    response = _session.get(api_url(port, "/api/stats"), timeout=REQUEST_TIMEOUT)
    assert_status(response, 200)
    stats = response.json()
    assert stats["requests_total"] > 0
    assert "user_count" in stats and "roles" in stats
    assert sum(stats["by_method"].values()) == stats["requests_total"]

    response = _session.get(
        api_url(port, "/openapi.json"), timeout=REQUEST_TIMEOUT
    )
    assert_status(response, 200)
    spec = response.json()
    assert spec["openapi"].startswith("3.")
    assert spec["servers"][0]["url"].endswith(f":{port}")

    custom = "abc-echo-xyz"
    response = _session.get(
        api_url(port, "/health"),
        headers={"X-Request-ID": custom},
        timeout=REQUEST_TIMEOUT,
    )
    assert response.headers.get("X-Request-ID") == custom
    assert get_users(port).headers.get("X-Request-ID")

    print("All automated API assertions passed.")


# ============================================================
# CLI / Entry Point
# ============================================================

def _parse_args(argv: Optional[list[str]] = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Local mock HTTP API simulator.",
    )
    parser.add_argument(
        "--port",
        type=int,
        default=None,
        help=(
            "Port to listen on. Defaults to MOCK_API_PORT, "
            f"then {DEFAULT_PORT}."
        ),
    )
    parser.add_argument(
        "--no-tests",
        action="store_true",
        help="Start the server without running the demo suite.",
    )
    parser.add_argument(
        "-v",
        "--verbose",
        action="store_true",
        help="Enable DEBUG-level logging.",
    )
    return parser.parse_args(argv)


def _resolve_port(cli_port: Optional[int]) -> int:
    if cli_port is not None:
        return cli_port
    env = os.environ.get("MOCK_API_PORT")
    if env:
        try:
            return int(env)
        except ValueError as exc:
            raise SystemExit(
                f"MOCK_API_PORT must be an integer, got {env!r}"
            ) from exc
    return DEFAULT_PORT


def _run_demo(port: int) -> None:
    print("=" * 60)
    print("MOCK API TEST")
    print("=" * 60)

    test_get_existing_user(port)
    test_get_nonexistent_user(port)
    test_get_invalid_user_id(port)

    print("\n--- Testing GET /api/users ---")
    list_users_as_table(port)

    test_create_user(port)
    print("\n--- Users After Creating Charlie ---")
    list_users_as_table(port)

    test_invalid_user(port)
    test_invalid_role(port)
    test_missing_name(port)
    test_missing_role(port)
    test_invalid_json(port)
    test_unsupported_media_type(port)

    test_replace_user(port)
    test_patch_user(port)
    test_delete_user(port)

    test_head_request(port)
    test_options_request(port)
    test_pagination(port)
    test_filter_role(port)
    test_search(port)
    test_health(port)
    test_stats(port)
    test_openapi(port)
    test_request_id_echo(port)
    test_method_not_allowed(port)

    print("\n--- Final User List ---")
    list_users_as_table(port)

    print("\n--- Resetting DB + stats before automated assertions ---")
    reset_database()
    _reset_stats()

    run_assertion_tests(port)


def main(argv: Optional[list[str]] = None) -> int:
    args = _parse_args(argv)

    logging.basicConfig(
        level=logging.DEBUG if args.verbose else logging.INFO,
        format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    )

    reset_database()
    _reset_stats()
    port = _resolve_port(args.port)

    ready = threading.Event()
    stop = threading.Event()

    server_thread = threading.Thread(
        target=run_server,
        args=(port, ready, stop),
        name="mock-api-server",
        daemon=True,
    )
    server_thread.start()

    if not ready.wait(timeout=5):
        log.error("Server failed to start within 5 seconds")
        return 1

    if not args.no_tests:
        try:
            _run_demo(port)
        except Exception:
            log.exception("Demo suite crashed")
            stop.set()
            server_thread.join(timeout=5)
            return 2

    log.info(
        "API simulator is running at http://%s:%d (Ctrl+C to stop)",
        HOST,
        port,
    )

    try:
        server_thread.join()
    except KeyboardInterrupt:
        log.info("Shutting down...")
    finally:
        stop.set()
        server_thread.join(timeout=5)

    return 0


def _get_free_port() -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("", 0))
        return s.getsockname()[1]


def _run_notebook_demo() -> int:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    )
    reset_database()
    _reset_stats()
    port = _get_free_port()

    ready = threading.Event()
    stop = threading.Event()
    threading.Thread(
        target=run_server,
        args=(port, ready, stop),
        name="mock-api-server",
        daemon=True,
    ).start()

    if not ready.wait(timeout=5):
        log.error("Server failed to start in notebook context")
        return 1

    print(f"Notebook mode: mock API listening on http://{HOST}:{port}")
    try:
        _run_demo(port)
    except Exception:
        log.exception("Demo suite crashed")
        return 2
    finally:
        stop.set()
    return 0


if __name__ == "__main__":
    if "ipykernel" in sys.modules or "google.colab" in sys.modules:
        _run_notebook_demo()
    else:
        sys.exit(main())


Notebook mode: mock API listening on http://localhost:47735
MOCK API TEST

--- Testing GET /api/users/1 ---
Status Code: 200
X-Request-ID: 7a6c0aa381ab4f02aec0a4b4f277d61a
Response: {'id': 1, 'name': 'Alice', 'role': 'Admin'}

--- Testing GET /api/users/99 ---
Status Code: 404
X-Request-ID: 367fa688031545039927784c07e28df3
Response: {'error': 'Not Found', 'message': 'User with ID 99 not found'}

--- Testing GET /api/users/invalid_id ---
Status Code: 400
X-Request-ID: 58bf0fb0cee94ad59b5843349a5a82a0
Response: {'error': 'Bad Request', 'message': 'User ID must be an integer'}

--- Testing GET /api/users ---
+--------+----------------------+-----------------+
| ID     | Name                 | Role            |
+--------+----------------------+-----------------+
| 1      | Alice                | Admin           |
| 2      | Bob                  | User            |
+--------+----------------------+-----------------+

--- Testing POST /api/users ---
Status Code: 201
X-Request-ID: 36d8a92927e